In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

MURA_ROOT = Path("/content/drive/MyDrive/MURA-v1.1")

print("Ruta:", MURA_ROOT)
print("¿Existe?:", MURA_ROOT.exists())

if not MURA_ROOT.exists():
    raise FileNotFoundError(f"No se encontró la carpeta: {MURA_ROOT}")

print("\nContenido raíz:\n")

for item in sorted(MURA_ROOT.iterdir()):
    tipo = "📁 CARPETA" if item.is_dir() else "📄 ARCHIVO"
    print(f"{tipo:12} {item.name}")

Ruta: /content/drive/MyDrive/MURA-v1.1
¿Existe?: True

Contenido raíz:

📁 CARPETA    train
📄 ARCHIVO    train_image_paths.csv
📄 ARCHIVO    train_labeled_studies.csv
📁 CARPETA    valid
📄 ARCHIVO    valid_image_paths.csv
📄 ARCHIVO    valid_labeled_studies.csv


In [ ]:
from pathlib import Path
import pandas as pd

MURA_ROOT = Path("/content/drive/MyDrive/MURA-v1.1")

# ============================================================
# 1. CONTAR IMÁGENES FÍSICAS
# ============================================================

extensiones = {".png", ".jpg", ".jpeg"}

train_images = [
    p for p in (MURA_ROOT / "train").rglob("*")
    if p.is_file() and p.suffix.lower() in extensiones
]

valid_images = [
    p for p in (MURA_ROOT / "valid").rglob("*")
    if p.is_file() and p.suffix.lower() in extensiones
]

print("=" * 70)
print("ARCHIVOS FÍSICOS")
print("=" * 70)

print(f"Imágenes TRAIN : {len(train_images):,}")
print(f"Imágenes VALID : {len(valid_images):,}")
print(f"TOTAL          : {len(train_images) + len(valid_images):,}")


# ============================================================
# 2. LEER LOS CSV OFICIALES
# ============================================================

train_paths = pd.read_csv(
    MURA_ROOT / "train_image_paths.csv",
    header=None,
    names=["path"]
)

valid_paths = pd.read_csv(
    MURA_ROOT / "valid_image_paths.csv",
    header=None,
    names=["path"]
)

train_studies = pd.read_csv(
    MURA_ROOT / "train_labeled_studies.csv",
    header=None,
    names=["study_path", "label"]
)

valid_studies = pd.read_csv(
    MURA_ROOT / "valid_labeled_studies.csv",
    header=None,
    names=["study_path", "label"]
)


print("\n" + "=" * 70)
print("CSV OFICIALES")
print("=" * 70)

print(f"Rutas TRAIN registradas : {len(train_paths):,}")
print(f"Rutas VALID registradas : {len(valid_paths):,}")
print(f"Total rutas             : {len(train_paths) + len(valid_paths):,}")

print()

print(f"Estudios TRAIN : {len(train_studies):,}")
print(f"Estudios VALID : {len(valid_studies):,}")
print(f"Total estudios : {len(train_studies) + len(valid_studies):,}")


# ============================================================
# 3. DISTRIBUCIÓN NORMAL / ABNORMAL A NIVEL DE ESTUDIO
# ============================================================

print("\n" + "=" * 70)
print("ETIQUETAS A NIVEL DE ESTUDIO")
print("=" * 70)

print("\nTRAIN:")
print(train_studies["label"].value_counts().sort_index())

print("\nVALID:")
print(valid_studies["label"].value_counts().sort_index())

print("\nInterpretación:")
print("0 = normal")
print("1 = abnormal")


# ============================================================
# 4. COMPROBAR COINCIDENCIA ENTRE CSV Y ARCHIVOS FÍSICOS
# ============================================================

print("\n" + "=" * 70)
print("COMPROBACIÓN CSV vs ARCHIVOS FÍSICOS")
print("=" * 70)

train_ok = len(train_images) == len(train_paths)
valid_ok = len(valid_images) == len(valid_paths)

print(
    f"TRAIN: {'✅ COINCIDE' if train_ok else '❌ NO COINCIDE'}"
)

print(
    f"VALID: {'✅ COINCIDE' if valid_ok else '❌ NO COINCIDE'}"
)


# ============================================================
# 5. REGIONES ANATÓMICAS
# ============================================================

def obtener_region(path):
    for parte in Path(path).parts:
        if parte.startswith("XR_"):
            return parte
    return None


train_paths["region"] = train_paths["path"].apply(obtener_region)
valid_paths["region"] = valid_paths["path"].apply(obtener_region)

print("\n" + "=" * 70)
print("REGIONES ANATÓMICAS")
print("=" * 70)

regiones = sorted(
    set(train_paths["region"].dropna()) |
    set(valid_paths["region"].dropna())
)

for region in regiones:
    print(region)

print(f"\nNúmero de regiones: {len(regiones)}")

ARCHIVOS FÍSICOS
Imágenes TRAIN : 36,812
Imágenes VALID : 3,197
TOTAL          : 40,009

CSV OFICIALES
Rutas TRAIN registradas : 36,808
Rutas VALID registradas : 3,197
Total rutas             : 40,005

Estudios TRAIN : 13,457
Estudios VALID : 1,199
Total estudios : 14,656

ETIQUETAS A NIVEL DE ESTUDIO

TRAIN:
label
0    8280
1    5177
Name: count, dtype: int64

VALID:
label
0    661
1    538
Name: count, dtype: int64

Interpretación:
0 = normal
1 = abnormal

COMPROBACIÓN CSV vs ARCHIVOS FÍSICOS
TRAIN: ❌ NO COINCIDE
VALID: ✅ COINCIDE

REGIONES ANATÓMICAS
XR_ELBOW
XR_FINGER
XR_FOREARM
XR_HAND
XR_HUMERUS
XR_SHOULDER
XR_WRIST

Número de regiones: 7


In [ ]:
from pathlib import Path
import pandas as pd

MURA_ROOT = Path("/content/drive/MyDrive/MURA-v1.1")

# ------------------------------------------------------------
# Función para normalizar las rutas del CSV
# ------------------------------------------------------------

def normalizar_ruta_csv(ruta):
    """
    Convierte una ruta del CSV de MURA a una ruta relativa
    respecto de MURA_ROOT.

    Ejemplo:
    MURA-v1.1/train/XR_WRIST/.../image1.png
                ↓
    train/XR_WRIST/.../image1.png
    """
    ruta = str(ruta).replace("\\", "/").strip()

    if ruta.startswith("MURA-v1.1/"):
        ruta = ruta[len("MURA-v1.1/"):]

    return ruta


# ------------------------------------------------------------
# 1. Rutas registradas oficialmente en los CSV
# ------------------------------------------------------------

train_csv = pd.read_csv(
    MURA_ROOT / "train_image_paths.csv",
    header=None,
    names=["path"]
)

valid_csv = pd.read_csv(
    MURA_ROOT / "valid_image_paths.csv",
    header=None,
    names=["path"]
)

train_csv_paths = {
    normalizar_ruta_csv(p)
    for p in train_csv["path"]
}

valid_csv_paths = {
    normalizar_ruta_csv(p)
    for p in valid_csv["path"]
}


# ------------------------------------------------------------
# 2. Rutas físicas encontradas en Google Drive
# ------------------------------------------------------------

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}

train_physical_paths = {
    p.relative_to(MURA_ROOT).as_posix()
    for p in (MURA_ROOT / "train").rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
}

valid_physical_paths = {
    p.relative_to(MURA_ROOT).as_posix()
    for p in (MURA_ROOT / "valid").rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
}


# ------------------------------------------------------------
# 3. Comparación
# ------------------------------------------------------------

extra_train = sorted(train_physical_paths - train_csv_paths)
missing_train = sorted(train_csv_paths - train_physical_paths)

extra_valid = sorted(valid_physical_paths - valid_csv_paths)
missing_valid = sorted(valid_csv_paths - valid_physical_paths)


# ------------------------------------------------------------
# 4. Resultados
# ------------------------------------------------------------

print("=" * 80)
print("TRAIN")
print("=" * 80)

print(f"\nImágenes físicas:       {len(train_physical_paths):,}")
print(f"Imágenes en CSV:        {len(train_csv_paths):,}")
print(f"Extras físicas:         {len(extra_train):,}")
print(f"Faltantes físicas:      {len(missing_train):,}")

print("\n--- ARCHIVOS EXTRA EN TRAIN ---")

if extra_train:
    for ruta in extra_train:
        print(ruta)
else:
    print("Ninguno")


print("\n--- ARCHIVOS DEL CSV QUE NO EXISTEN FÍSICAMENTE ---")

if missing_train:
    for ruta in missing_train:
        print(ruta)
else:
    print("Ninguno")


print("\n" + "=" * 80)
print("VALID")
print("=" * 80)

print(f"\nImágenes físicas:       {len(valid_physical_paths):,}")
print(f"Imágenes en CSV:        {len(valid_csv_paths):,}")
print(f"Extras físicas:         {len(extra_valid):,}")
print(f"Faltantes físicas:      {len(missing_valid):,}")

print("\n--- ARCHIVOS EXTRA EN VALID ---")

if extra_valid:
    for ruta in extra_valid:
        print(ruta)
else:
    print("Ninguno")

print("\n--- ARCHIVOS DEL CSV QUE NO EXISTEN FÍSICAMENTE ---")

if missing_valid:
    for ruta in missing_valid:
        print(ruta)
else:
    print("Ninguno")

TRAIN

Imágenes físicas:       36,812
Imágenes en CSV:        36,808
Extras físicas:         4
Faltantes físicas:      0

--- ARCHIVOS EXTRA EN TRAIN ---
train/XR_WRIST/patient07840/study1_negative/._image1.png
train/XR_WRIST/patient07840/study2_negative/._image1.png
train/XR_WRIST/patient07840/study2_negative/._image2.png
train/XR_WRIST/patient07840/study2_negative/._image3.png

--- ARCHIVOS DEL CSV QUE NO EXISTEN FÍSICAMENTE ---
Ninguno

VALID

Imágenes físicas:       3,197
Imágenes en CSV:        3,197
Extras físicas:         0
Faltantes físicas:      0

--- ARCHIVOS EXTRA EN VALID ---
Ninguno

--- ARCHIVOS DEL CSV QUE NO EXISTEN FÍSICAMENTE ---
Ninguno


In [ ]:
from pathlib import Path
import pandas as pd

MURA_ROOT = Path("/content/drive/MyDrive/MURA-v1.1")

# ============================================================
# 1. LEER CSV OFICIALES
# ============================================================

train_paths = pd.read_csv(
    MURA_ROOT / "train_image_paths.csv",
    header=None,
    names=["image_path_csv"]
)

valid_paths = pd.read_csv(
    MURA_ROOT / "valid_image_paths.csv",
    header=None,
    names=["image_path_csv"]
)

train_studies = pd.read_csv(
    MURA_ROOT / "train_labeled_studies.csv",
    header=None,
    names=["study_path_csv", "label"]
)

valid_studies = pd.read_csv(
    MURA_ROOT / "valid_labeled_studies.csv",
    header=None,
    names=["study_path_csv", "label"]
)


# ============================================================
# 2. FUNCIONES PARA INTERPRETAR LAS RUTAS DE MURA
# ============================================================

def normalizar(ruta):
    return str(ruta).replace("\\", "/").strip()


def extraer_metadata(ruta):
    """
    Ejemplo esperado:

    MURA-v1.1/train/XR_WRIST/patient00001/
    study1_positive/image1.png
    """

    ruta = normalizar(ruta)
    partes = ruta.split("/")

    region = next(
        (p for p in partes if p.startswith("XR_")),
        None
    )

    patient = next(
        (p for p in partes if p.startswith("patient")),
        None
    )

    study = next(
        (p for p in partes
         if p.startswith("study") and
         ("positive" in p or "negative" in p)),
        None
    )

    if study is None:
        label_folder = None
    elif "positive" in study:
        label_folder = 1
    elif "negative" in study:
        label_folder = 0
    else:
        label_folder = None

    image_name = partes[-1]

    return pd.Series({
        "region": region,
        "patient": patient,
        "study": study,
        "label_folder": label_folder,
        "image_name": image_name
    })


# ============================================================
# 3. CREAR DATAFRAME DE IMÁGENES
# ============================================================

train_meta = train_paths["image_path_csv"].apply(extraer_metadata)
valid_meta = valid_paths["image_path_csv"].apply(extraer_metadata)

train_df = pd.concat(
    [train_paths, train_meta],
    axis=1
)

valid_df = pd.concat(
    [valid_paths, valid_meta],
    axis=1
)

train_df["split"] = "train"
valid_df["split"] = "valid"


# ============================================================
# 4. CREAR RUTA LOCAL REAL EN GOOGLE DRIVE
# ============================================================

def convertir_a_ruta_drive(ruta):
    ruta = normalizar(ruta)

    if ruta.startswith("MURA-v1.1/"):
        ruta = ruta[len("MURA-v1.1/"):]

    return str(MURA_ROOT / ruta)


train_df["image_path"] = (
    train_df["image_path_csv"].apply(convertir_a_ruta_drive)
)

valid_df["image_path"] = (
    valid_df["image_path_csv"].apply(convertir_a_ruta_drive)
)


# ============================================================
# 5. UNIR TRAIN + VALID
# ============================================================

df = pd.concat(
    [train_df, valid_df],
    ignore_index=True
)


# ============================================================
# 6. IDENTIFICADOR ÚNICO DEL ESTUDIO
# ============================================================

df["study_id"] = (
    df["split"] + "/" +
    df["region"].astype(str) + "/" +
    df["patient"].astype(str) + "/" +
    df["study"].astype(str)
)


# ============================================================
# 7. ETIQUETA LEGIBLE
# ============================================================

df["label"] = df["label_folder"]

df["class_name"] = df["label"].map({
    0: "normal",
    1: "abnormal"
})


# ============================================================
# 8. VALIDACIONES BÁSICAS
# ============================================================

print("=" * 75)
print("DATAFRAME MAESTRO")
print("=" * 75)

print(f"\nTotal de radiografías: {len(df):,}")
print(f"Total de estudios:     {df['study_id'].nunique():,}")

print("\nValores faltantes importantes:")

print(
    df[
        ["region", "patient", "study", "label"]
    ].isna().sum()
)


# ============================================================
# 9. DISTRIBUCIÓN NORMAL / ABNORMAL
# ============================================================

print("\n" + "=" * 75)
print("DISTRIBUCIÓN DE RADIOGRAFÍAS")
print("=" * 75)

print(
    pd.crosstab(
        df["split"],
        df["class_name"],
        margins=True
    )
)


# ============================================================
# 10. QUEDARNOS CON ABNORMAL
# ============================================================

abnormal_df = (
    df[df["label"] == 1]
    .copy()
    .reset_index(drop=True)
)


print("\n" + "=" * 75)
print("RADIOGRAFÍAS ABNORMAL")
print("=" * 75)

print(f"\nTotal abnormal: {len(abnormal_df):,}")

print("\nPor split:")
print(
    abnormal_df["split"]
    .value_counts()
)


# ============================================================
# 11. DISTRIBUCIÓN ABNORMAL POR REGIÓN
# ============================================================

print("\n" + "=" * 75)
print("ABNORMAL POR REGIÓN")
print("=" * 75)

region_abnormal = pd.crosstab(
    abnormal_df["region"],
    abnormal_df["split"],
    margins=True
)

print(region_abnormal)


# ============================================================
# 12. NÚMERO DE ESTUDIOS ABNORMAL POR REGIÓN
# ============================================================

print("\n" + "=" * 75)
print("ESTUDIOS ABNORMAL POR REGIÓN")
print("=" * 75)

studies_by_region = (
    abnormal_df
    .groupby(["region", "split"])["study_id"]
    .nunique()
    .unstack(fill_value=0)
)

studies_by_region["Total"] = studies_by_region.sum(axis=1)

print(studies_by_region)


# ============================================================
# 13. VISTA PREVIA
# ============================================================

print("\n" + "=" * 75)
print("EJEMPLO DEL DATAFRAME")
print("=" * 75)

display(
    abnormal_df[
        [
            "split",
            "region",
            "patient",
            "study",
            "image_name",
            "class_name",
            "image_path"
        ]
    ].head(10)
)

DATAFRAME MAESTRO

Total de radiografías: 40,005
Total de estudios:     14,656

Valores faltantes importantes:
region     0
patient    0
study      0
label      0
dtype: int64

DISTRIBUCIÓN DE RADIOGRAFÍAS
class_name  abnormal  normal    All
split                              
train          14873   21935  36808
valid           1530    1667   3197
All            16403   23602  40005

RADIOGRAFÍAS ABNORMAL

Total abnormal: 16,403

Por split:
split
train    14873
valid     1530
Name: count, dtype: int64

ABNORMAL POR REGIÓN
split        train  valid    All
region                          
XR_ELBOW      2006    230   2236
XR_FINGER     1968    247   2215
XR_FOREARM     661    151    812
XR_HAND       1484    189   1673
XR_HUMERUS     599    140    739
XR_SHOULDER   4168    278   4446
XR_WRIST      3987    295   4282
All          14873   1530  16403

ESTUDIOS ABNORMAL POR REGIÓN
split        train  valid  Total
region                          
XR_ELBOW       660     66    726
XR_FINGER    

,split,region,patient,study,image_name,class_name,image_path
0,train,XR_SHOULDER,patient00001,study1_positive,image1.png,abnormal,/content/drive/MyDrive/MURA-v1.1/train/XR_SHOU...
1,train,XR_SHOULDER,patient00001,study1_positive,image2.png,abnormal,/content/drive/MyDrive/MURA-v1.1/train/XR_SHOU...
2,train,XR_SHOULDER,patient00001,study1_positive,image3.png,abnormal,/content/drive/MyDrive/MURA-v1.1/train/XR_SHOU...
3,train,XR_SHOULDER,patient00002,study1_positive,image1.png,abnormal,/content/drive/MyDrive/MURA-v1.1/train/XR_SHOU...
4,train,XR_SHOULDER,patient00002,study1_positive,image2.png,abnormal,/content/drive/MyDrive/MURA-v1.1/train/XR_SHOU...
5,train,XR_SHOULDER,patient00002,study1_positive,image3.png,abnormal,/content/drive/MyDrive/MURA-v1.1/train/XR_SHOU...
6,train,XR_SHOULDER,patient00003,study1_positive,image1.png,abnormal,/content/drive/MyDrive/MURA-v1.1/train/XR_SHOU...
7,train,XR_SHOULDER,patient00003,study1_positive,image2.png,abnormal,/content/drive/MyDrive/MURA-v1.1/train/XR_SHOU...
8,train,XR_SHOULDER,patient00003,study1_positive,image3.png,abnormal,/content/drive/MyDrive/MURA-v1.1/train/XR_SHOU...
9,train,XR_SHOULDER,patient00004,study1_positive,image1.png,abnormal,/content/drive/MyDrive/MURA-v1.1/train/XR_SHOU...


In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# CARPETA DE TRABAJO DEL PROYECTO
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MURA_TT"
)

METADATA_DIR = PROJECT_ROOT / "metadata"

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. GUARDAR DATAFRAME MAESTRO
# ============================================================

df.to_csv(
    METADATA_DIR / "mura_master.csv",
    index=False
)


# ============================================================
# 2. GUARDAR TODAS LAS IMÁGENES ABNORMAL
# ============================================================

abnormal_df.to_csv(
    METADATA_DIR / "mura_abnormal_all.csv",
    index=False
)


# ============================================================
# 3. DATASET QUE UTILIZAREMOS PARA CLUSTERING
#    SOLO ABNORMAL + TRAIN
# ============================================================

cluster_df = (
    abnormal_df[
        abnormal_df["split"] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)


cluster_df.to_csv(
    METADATA_DIR / "mura_abnormal_train.csv",
    index=False
)


# ============================================================
# 4. VERIFICACIÓN
# ============================================================

print("=" * 70)
print("ARCHIVOS GUARDADOS")
print("=" * 70)

print(
    METADATA_DIR / "mura_master.csv",
    f"→ {len(df):,} imágenes"
)

print(
    METADATA_DIR / "mura_abnormal_all.csv",
    f"→ {len(abnormal_df):,} imágenes"
)

print(
    METADATA_DIR / "mura_abnormal_train.csv",
    f"→ {len(cluster_df):,} imágenes"
)


print("\n" + "=" * 70)
print("CONJUNTO PARA CLUSTERING")
print("=" * 70)

print(f"\nRadiografías: {len(cluster_df):,}")
print(
    f"Estudios:     {cluster_df['study_id'].nunique():,}"
)

print("\nDistribución por región:")

print(
    cluster_df["region"]
    .value_counts()
    .sort_index()
)


print("\n" + "=" * 70)
print("VALID RESERVADO")
print("=" * 70)

valid_abnormal = abnormal_df[
    abnormal_df["split"] == "valid"
]

print(
    f"\nRadiografías abnormal reservadas: "
    f"{len(valid_abnormal):,}"
)

print(
    f"Estudios abnormal reservados: "
    f"{valid_abnormal['study_id'].nunique():,}"
)

ARCHIVOS GUARDADOS
/content/drive/MyDrive/MURA_TT/metadata/mura_master.csv → 40,005 imágenes
/content/drive/MyDrive/MURA_TT/metadata/mura_abnormal_all.csv → 16,403 imágenes
/content/drive/MyDrive/MURA_TT/metadata/mura_abnormal_train.csv → 14,873 imágenes

CONJUNTO PARA CLUSTERING

Radiografías: 14,873
Estudios:     5,177

Distribución por región:
region
XR_ELBOW       2006
XR_FINGER      1968
XR_FOREARM      661
XR_HAND        1484
XR_HUMERUS      599
XR_SHOULDER    4168
XR_WRIST       3987
Name: count, dtype: int64

VALID RESERVADO

Radiografías abnormal reservadas: 1,530
Estudios abnormal reservados: 538


In [ ]:
from pathlib import Path
from shutil import copy2
from tqdm.auto import tqdm
import pandas as pd
import time

# ============================================================
# CONFIGURACIÓN
# ============================================================

METADATA_FILE = Path(
    "/content/drive/MyDrive/MURA_TT/metadata/mura_abnormal_train.csv"
)

LOCAL_ROOT = Path("/content/MURA_abnormal_train")

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. CARGAR METADATOS
# ============================================================

cluster_df = pd.read_csv(METADATA_FILE)

print(f"Radiografías a copiar: {len(cluster_df):,}")
print(f"Destino local: {LOCAL_ROOT}")


# ============================================================
# 2. COPIAR IMÁGENES
#
# Conservamos:
# región / paciente / estudio / imagen
# ============================================================

errores = []

inicio = time.time()

for row in tqdm(
    cluster_df.itertuples(index=False),
    total=len(cluster_df),
    desc="Copiando radiografías"
):

    origen = Path(row.image_path)

    destino = (
        LOCAL_ROOT
        / row.region
        / row.patient
        / row.study
        / row.image_name
    )

    # Permite reanudar la celda sin volver a copiar
    # imágenes que ya existen.
    if destino.exists():
        continue

    try:
        destino.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        copy2(origen, destino)

    except Exception as e:
        errores.append({
            "origen": str(origen),
            "error": str(e)
        })


# ============================================================
# 3. CREAR RUTA LOCAL EN EL DATAFRAME
# ============================================================

cluster_df["local_path"] = cluster_df.apply(
    lambda row: str(
        LOCAL_ROOT
        / row["region"]
        / row["patient"]
        / row["study"]
        / row["image_name"]
    ),
    axis=1
)


# ============================================================
# 4. VERIFICAR SOLO LAS RUTAS ESPERADAS
#
# IMPORTANTE:
# No usamos rglob().
# ============================================================

cluster_df["local_exists"] = (
    cluster_df["local_path"]
    .apply(lambda x: Path(x).is_file())
)

n_ok = int(cluster_df["local_exists"].sum())
n_missing = len(cluster_df) - n_ok


# ============================================================
# 5. TIEMPO
# ============================================================

fin = time.time()

minutos = (fin - inicio) / 60


# ============================================================
# 6. RESULTADOS
# ============================================================

print("\n" + "=" * 70)
print("COPIA TERMINADA")
print("=" * 70)

print(f"\nEsperadas:        {len(cluster_df):,}")
print(f"Copiadas/locales: {n_ok:,}")
print(f"Faltantes:        {n_missing:,}")
print(f"Errores:          {len(errores):,}")

print(f"\nTiempo: {minutos:.2f} minutos")


# ============================================================
# 7. MOSTRAR ERRORES SI EXISTEN
# ============================================================

if errores:

    errores_df = pd.DataFrame(errores)

    print("\nPrimeros errores:")

    display(
        errores_df.head(20)
    )

else:

    print("\n✅ No se registraron errores.")


# ============================================================
# 8. GUARDAR MANIFEST ACTUALIZADO
# ============================================================

cluster_df.to_csv(
    "/content/drive/MyDrive/MURA_TT/metadata/"
    "mura_abnormal_train_local_manifest.csv",
    index=False
)

print(
    "\nManifest guardado correctamente."
)

Radiografías a copiar: 14,873
Destino local: /content/MURA_abnormal_train


Copiando radiografías:   0%|          | 0/14873 [00:00<?, ?it/s]


COPIA TERMINADA

Esperadas:        14,873
Copiadas/locales: 14,873
Faltantes:        0
Errores:          0

Tiempo: 102.61 minutos

✅ No se registraron errores.

Manifest guardado correctamente.


In [ ]:
import torch
import torchvision

print("=" * 70)
print("ENTORNO")
print("=" * 70)

print(f"PyTorch:     {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA:        {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:         {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM:        "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )
else:
    print("\n⚠️ No se detectó GPU.")
    print("Activa una GPU antes de continuar.")

ENTORNO
PyTorch:     2.11.0+cu130
Torchvision: 0.26.0+cu130
CUDA:        True
GPU:         Tesla T4
VRAM:        14.56 GB


In [ ]:
import torch
import torch.nn as nn
from torchvision.models import (
    densenet121,
    DenseNet121_Weights
)

# ============================================================
# DISPOSITIVO
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Dispositivo: {device}")


# ============================================================
# DENSENET121 PREENTRENADA EN IMAGENET
# ============================================================

weights = DenseNet121_Weights.DEFAULT

model = densenet121(weights=weights)


# ============================================================
# QUITAR CLASIFICADOR
#
# DenseNet121 produce 1024 características antes de
# la capa final de clasificación.
# ============================================================

model.classifier = nn.Identity()

model = model.to(device)
model.eval()


# ============================================================
# TRANSFORMACIÓN ASOCIADA A LOS PESOS
# ============================================================

preprocess = weights.transforms()


print("\n" + "=" * 70)
print("MODELO")
print("=" * 70)

print("Arquitectura: DenseNet121")
print("Pesos:        ImageNet pretrained")
print("Clasificador: eliminado")
print("Embedding:    1024 características")
print(f"Dispositivo:  {device}")

Dispositivo: cuda
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 82.6MB/s]



MODELO
Arquitectura: DenseNet121
Pesos:        ImageNet pretrained
Clasificador: eliminado
Embedding:    1024 características
Dispositivo:  cuda


In [ ]:
from pathlib import Path
from PIL import Image
import torch
import numpy as np
import pandas as pd
import time

# ============================================================
# CONFIGURACIÓN
# ============================================================

N_PRUEBA = 32

test_df = cluster_df.head(N_PRUEBA).copy()

print(f"Imágenes de prueba: {len(test_df)}")


# ============================================================
# CARGAR Y PREPROCESAR IMÁGENES
# ============================================================

imagenes = []
errores = []

inicio = time.time()

for row in test_df.itertuples(index=False):

    try:
        # Estamos usando la copia LOCAL, no Google Drive
        image = Image.open(row.local_path)

        # DenseNet preentrenada espera 3 canales
        image = image.convert("RGB")

        # Transformaciones oficiales asociadas a los pesos
        tensor = preprocess(image)

        imagenes.append(tensor)

        image.close()

    except Exception as e:

        errores.append({
            "path": row.local_path,
            "error": str(e)
        })


# ============================================================
# CREAR BATCH
# ============================================================

batch = torch.stack(imagenes)

print("\nForma del batch:")
print(batch.shape)


# ============================================================
# MOVER A GPU
# ============================================================

batch = batch.to(device)


# ============================================================
# EXTRAER EMBEDDINGS
# ============================================================

with torch.inference_mode():

    embeddings_test = model(batch)


# ============================================================
# PASAR RESULTADOS A CPU
# ============================================================

embeddings_test = (
    embeddings_test
    .cpu()
    .numpy()
)


# ============================================================
# RESULTADOS
# ============================================================

fin = time.time()

print("\n" + "=" * 70)
print("PRUEBA DE EMBEDDINGS")
print("=" * 70)

print(
    f"\nImágenes procesadas: "
    f"{embeddings_test.shape[0]}"
)

print(
    f"Características por imagen: "
    f"{embeddings_test.shape[1]}"
)

print(
    f"Forma final: "
    f"{embeddings_test.shape}"
)

print(
    f"Tipo de datos: "
    f"{embeddings_test.dtype}"
)

print(
    f"Tiempo: "
    f"{fin - inicio:.2f} segundos"
)

print(
    f"Errores: "
    f"{len(errores)}"
)


# ============================================================
# VALIDAR NÚMEROS
# ============================================================

print("\n" + "=" * 70)
print("VALIDACIÓN NUMÉRICA")
print("=" * 70)

print(
    "NaN:",
    np.isnan(embeddings_test).sum()
)

print(
    "Inf:",
    np.isinf(embeddings_test).sum()
)

print(
    "Mínimo:",
    embeddings_test.min()
)

print(
    "Máximo:",
    embeddings_test.max()
)

print(
    "Media:",
    embeddings_test.mean()
)

print(
    "Desviación estándar:",
    embeddings_test.std()
)

Imágenes de prueba: 32

Forma del batch:
torch.Size([32, 3, 224, 224])

PRUEBA DE EMBEDDINGS

Imágenes procesadas: 32
Características por imagen: 1024
Forma final: (32, 1024)
Tipo de datos: float32
Tiempo: 2.83 segundos
Errores: 0

VALIDACIÓN NUMÉRICA
NaN: 0
Inf: 0
Mínimo: 0.0
Máximo: 9.248166
Media: 0.38263518
Desviación estándar: 0.72454196


In [ ]:
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import torch
import numpy as np
import pandas as pd
import time


# ============================================================
# CONFIGURACIÓN
# ============================================================

MANIFEST_FILE = Path(
    "/content/drive/MyDrive/MURA_TT/metadata/"
    "mura_abnormal_train_local_manifest.csv"
)

EMBEDDINGS_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/embeddings"
)

EMBEDDINGS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BATCH_SIZE = 64
NUM_WORKERS = 2


# ============================================================
# 1. CARGAR MANIFEST
# ============================================================

cluster_df = pd.read_csv(MANIFEST_FILE)

print("=" * 70)
print("CONFIGURACIÓN")
print("=" * 70)

print(f"\nRadiografías: {len(cluster_df):,}")
print(f"Batch size:   {BATCH_SIZE}")
print(f"Workers:      {NUM_WORKERS}")
print(f"Dispositivo:  {device}")


# ============================================================
# 2. DATASET
# ============================================================

class MURAEmbeddingDataset(Dataset):

    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        path = self.df.iloc[idx]["local_path"]

        with Image.open(path) as image:

            image = image.convert("RGB")

            tensor = self.transform(image)

        return tensor, idx


dataset = MURAEmbeddingDataset(
    cluster_df,
    preprocess
)


# ============================================================
# 3. DATALOADER
# ============================================================

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,       # MUY IMPORTANTE
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)


# ============================================================
# 4. MATRIZ DONDE GUARDAREMOS LOS EMBEDDINGS
#
# 14,873 x 1,024 float32 ≈ 58 MB
# ============================================================

all_embeddings = np.empty(
    (len(dataset), 1024),
    dtype=np.float32
)


# ============================================================
# 5. EXTRACCIÓN
# ============================================================

model.eval()

inicio = time.time()

procesadas = 0


with torch.inference_mode():

    for images, indices in tqdm(
        loader,
        desc="Extrayendo embeddings"
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        outputs = model(images)

        outputs = (
            outputs
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        indices_np = indices.numpy()

        all_embeddings[indices_np] = outputs

        procesadas += len(indices_np)


tiempo_total = time.time() - inicio


# ============================================================
# 6. VALIDACIÓN
# ============================================================

print("\n" + "=" * 70)
print("VALIDACIÓN")
print("=" * 70)

print(f"\nProcesadas: {procesadas:,}")

print(
    f"Forma embeddings: "
    f"{all_embeddings.shape}"
)

print(
    f"Tipo: "
    f"{all_embeddings.dtype}"
)

print(
    f"NaN: "
    f"{np.isnan(all_embeddings).sum():,}"
)

print(
    f"Inf: "
    f"{np.isinf(all_embeddings).sum():,}"
)

print(
    f"Mínimo: "
    f"{all_embeddings.min():.6f}"
)

print(
    f"Máximo: "
    f"{all_embeddings.max():.6f}"
)

print(
    f"Media: "
    f"{all_embeddings.mean():.6f}"
)

print(
    f"Desv. estándar: "
    f"{all_embeddings.std():.6f}"
)

print(
    f"\nTiempo total: "
    f"{tiempo_total / 60:.2f} minutos"
)

print(
    f"Velocidad: "
    f"{len(dataset) / tiempo_total:.2f} imágenes/s"
)


# ============================================================
# 7. GUARDAR EMBEDDINGS EN DRIVE
# ============================================================

EMBEDDINGS_FILE = (
    EMBEDDINGS_DIR /
    "densenet121_imagenet_abnormal_train.npy"
)

np.save(
    EMBEDDINGS_FILE,
    all_embeddings
)


# ============================================================
# 8. GUARDAR MANIFEST EN EL MISMO ORDEN
#
# embedding_index nos permite saber exactamente qué fila
# de la matriz corresponde a qué radiografía.
# ============================================================

embedding_manifest = cluster_df.copy()

embedding_manifest["embedding_index"] = np.arange(
    len(embedding_manifest)
)

MANIFEST_OUT = (
    EMBEDDINGS_DIR /
    "densenet121_imagenet_abnormal_train_manifest.csv"
)

embedding_manifest.to_csv(
    MANIFEST_OUT,
    index=False
)


# ============================================================
# 9. COMPROBAR QUE SE GUARDARON
# ============================================================

print("\n" + "=" * 70)
print("ARCHIVOS GUARDADOS")
print("=" * 70)

print(f"\nEmbeddings:")
print(EMBEDDINGS_FILE)

print(f"\nManifest:")
print(MANIFEST_OUT)

print(
    f"\nTamaño embeddings: "
    f"{EMBEDDINGS_FILE.stat().st_size / 1024**2:.2f} MB"
)

print("\n✅ Extracción terminada y guardada.")

CONFIGURACIÓN

Radiografías: 14,873
Batch size:   64
Workers:      2
Dispositivo:  cuda


Extrayendo embeddings:   0%|          | 0/233 [00:00<?, ?it/s]


VALIDACIÓN

Procesadas: 14,873
Forma embeddings: (14873, 1024)
Tipo: float32
NaN: 0
Inf: 0
Mínimo: 0.000000
Máximo: 16.781796
Media: 0.412056
Desv. estándar: 0.779591

Tiempo total: 1.83 minutos
Velocidad: 135.09 imágenes/s

ARCHIVOS GUARDADOS

Embeddings:
/content/drive/MyDrive/MURA_TT/embeddings/densenet121_imagenet_abnormal_train.npy

Manifest:
/content/drive/MyDrive/MURA_TT/embeddings/densenet121_imagenet_abnormal_train_manifest.csv

Tamaño embeddings: 58.10 MB

✅ Extracción terminada y guardada.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import time


# ============================================================
# RUTAS
# ============================================================

EMBEDDINGS_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/embeddings"
)

PCA_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/pca"
)

PCA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


EMBEDDINGS_FILE = (
    EMBDINGS_DIR
    if False else
    EMBEDDINGS_DIR / "densenet121_imagenet_abnormal_train.npy"
)

MANIFEST_FILE = (
    EMBEDDINGS_DIR /
    "densenet121_imagenet_abnormal_train_manifest.csv"
)


# ============================================================
# 1. CARGAR EMBEDDINGS Y MANIFEST
# ============================================================

embeddings = np.load(EMBEDDINGS_FILE)

manifest = pd.read_csv(MANIFEST_FILE)


print("=" * 70)
print("DATOS CARGADOS")
print("=" * 70)

print(f"\nEmbeddings: {embeddings.shape}")
print(f"Manifest:   {manifest.shape}")


assert len(embeddings) == len(manifest)

assert (
    manifest["embedding_index"].to_numpy()
    == np.arange(len(manifest))
).all()

print("\n✅ Correspondencia embedding ↔ imagen correcta.")


# ============================================================
# 2. REGIONES
# ============================================================

regiones = sorted(
    manifest["region"].unique()
)

print("\nRegiones:")

for region in regiones:
    print(
        f"{region}: "
        f"{(manifest['region'] == region).sum():,}"
    )


# ============================================================
# 3. PCA INDEPENDIENTE POR REGIÓN
# ============================================================

resultados_pca = []

pca_data = {}

inicio_total = time.time()


for region in regiones:

    print("\n" + "=" * 70)
    print(region)
    print("=" * 70)

    mask = (
        manifest["region"] == region
    ).to_numpy()

    indices = np.where(mask)[0]

    X = embeddings[indices]


    # --------------------------------------------------------
    # ESTANDARIZACIÓN
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)


    # --------------------------------------------------------
    # PCA
    #
    # n_components=0.95:
    # conservar suficientes componentes para explicar
    # el 95 % de la varianza.
    # --------------------------------------------------------

    pca = PCA(
        n_components=0.95,
        svd_solver="full"
    )

    inicio = time.time()

    X_pca = pca.fit_transform(
        X_scaled
    )

    tiempo = time.time() - inicio


    # --------------------------------------------------------
    # GUARDAR EN MEMORIA
    # --------------------------------------------------------

    pca_data[region] = {
        "X_pca": X_pca.astype(np.float32),
        "indices": indices,
        "scaler": scaler,
        "pca": pca
    }


    # --------------------------------------------------------
    # RESULTADOS
    # --------------------------------------------------------

    n_original = X.shape[1]

    n_pca = X_pca.shape[1]

    varianza = (
        pca.explained_variance_ratio_.sum()
    )


    resultados_pca.append({
        "region": region,
        "n_images": len(X),
        "original_dimensions": n_original,
        "pca_dimensions": n_pca,
        "variance_explained": varianza,
        "time_seconds": tiempo
    })


    print(
        f"Imágenes:              {len(X):,}"
    )

    print(
        f"Dimensiones originales: {n_original}"
    )

    print(
        f"Dimensiones PCA:        {n_pca}"
    )

    print(
        f"Varianza conservada:    "
        f"{varianza * 100:.2f}%"
    )

    print(
        f"Tiempo PCA:             "
        f"{tiempo:.2f} s"
    )


# ============================================================
# 4. TABLA RESUMEN
# ============================================================

pca_summary = pd.DataFrame(
    resultados_pca
)

print("\n" + "=" * 70)
print("RESUMEN PCA")
print("=" * 70)

display(pca_summary)


# ============================================================
# 5. GUARDAR RESULTADOS PCA
#
# Guardamos únicamente matrices e índices.
# Los objetos scaler/PCA no son necesarios para hacer
# el clustering de esta muestra.
# ============================================================

for region in regiones:

    safe_region = region.lower()

    np.savez_compressed(
        PCA_DIR / f"{safe_region}_pca.npz",

        X_pca=pca_data[region]["X_pca"],

        indices=pca_data[region]["indices"]
    )


pca_summary.to_csv(
    PCA_DIR / "pca_summary.csv",
    index=False
)


tiempo_total = time.time() - inicio_total

print(
    f"\nTiempo total PCA: "
    f"{tiempo_total:.2f} segundos"
)

print(
    f"\nResultados guardados en:\n"
    f"{PCA_DIR}"
)

print("\n✅ PCA terminado.")

DATOS CARGADOS

Embeddings: (14873, 1024)
Manifest:   (14873, 14)

✅ Correspondencia embedding ↔ imagen correcta.

Regiones:
XR_ELBOW: 2,006
XR_FINGER: 1,968
XR_FOREARM: 661
XR_HAND: 1,484
XR_HUMERUS: 599
XR_SHOULDER: 4,168
XR_WRIST: 3,987

XR_ELBOW
Imágenes:              2,006
Dimensiones originales: 1024
Dimensiones PCA:        372
Varianza conservada:    95.01%
Tiempo PCA:             0.73 s

XR_FINGER
Imágenes:              1,968
Dimensiones originales: 1024
Dimensiones PCA:        362
Varianza conservada:    95.02%
Tiempo PCA:             0.74 s

XR_FOREARM
Imágenes:              661
Dimensiones originales: 1024
Dimensiones PCA:        261
Varianza conservada:    95.01%
Tiempo PCA:             0.20 s

XR_HAND
Imágenes:              1,484
Dimensiones originales: 1024
Dimensiones PCA:        338
Varianza conservada:    95.02%
Tiempo PCA:             0.60 s

XR_HUMERUS
Imágenes:              599
Dimensiones originales: 1024
Dimensiones PCA:        251
Varianza conservada:    95.02%
T

,region,n_images,original_dimensions,pca_dimensions,variance_explained,time_seconds
0,XR_ELBOW,2006,1024,372,0.950096,0.726412
1,XR_FINGER,1968,1024,362,0.950216,0.743295
2,XR_FOREARM,661,1024,261,0.950136,0.201912
3,XR_HAND,1484,1024,338,0.950247,0.596305
4,XR_HUMERUS,599,1024,251,0.950237,0.174044
5,XR_SHOULDER,4168,1024,428,0.950123,1.027495
6,XR_WRIST,3987,1024,412,0.950146,0.934543



Tiempo total PCA: 5.96 segundos

Resultados guardados en:
/content/drive/MyDrive/MURA_TT/pca

✅ PCA terminado.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import time

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

from tqdm.auto import tqdm


# ============================================================
# CONFIGURACIÓN
# ============================================================

PCA_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/pca"
)

CLUSTER_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/clustering"
)

CLUSTER_DIR.mkdir(
    parents=True,
    exist_ok=True
)


K_VALUES = range(2, 31)

RANDOM_STATE = 42


# ============================================================
# REGIONES
# ============================================================

regiones = [
    "XR_ELBOW",
    "XR_FINGER",
    "XR_FOREARM",
    "XR_HAND",
    "XR_HUMERUS",
    "XR_SHOULDER",
    "XR_WRIST"
]


# ============================================================
# RESULTADOS
# ============================================================

todos_resultados = []

inicio_total = time.time()


# ============================================================
# EVALUAR CADA REGIÓN
# ============================================================

for region in regiones:

    print("\n" + "=" * 75)
    print(region)
    print("=" * 75)

    archivo = (
        PCA_DIR /
        f"{region.lower()}_pca.npz"
    )

    datos = np.load(archivo)

    X = datos["X_pca"]

    print(
        f"Imágenes: {X.shape[0]:,} | "
        f"Dimensiones PCA: {X.shape[1]}"
    )

    resultados_region = []


    # ========================================================
    # PROBAR DIFERENTES K
    # ========================================================

    for k in tqdm(
        K_VALUES,
        desc=f"Evaluando {region}"
    ):

        inicio = time.time()


        # ----------------------------------------------------
        # K-MEANS
        # ----------------------------------------------------

        kmeans = KMeans(
            n_clusters=k,
            init="k-means++",
            n_init=10,
            max_iter=300,
            random_state=RANDOM_STATE
        )

        labels = kmeans.fit_predict(X)


        # ----------------------------------------------------
        # TAMAÑOS DE CLUSTERS
        # ----------------------------------------------------

        cluster_sizes = np.bincount(
            labels,
            minlength=k
        )

        min_cluster = int(
            cluster_sizes.min()
        )

        max_cluster = int(
            cluster_sizes.max()
        )

        min_cluster_pct = (
            min_cluster /
            len(X) *
            100
        )


        # ----------------------------------------------------
        # MÉTRICAS
        # ----------------------------------------------------

        silhouette = silhouette_score(
            X,
            labels,
            metric="euclidean"
        )

        davies = davies_bouldin_score(
            X,
            labels
        )

        calinski = calinski_harabasz_score(
            X,
            labels
        )


        tiempo = time.time() - inicio


        resultado = {
            "region": region,
            "k": k,

            "silhouette": silhouette,

            "davies_bouldin": davies,

            "calinski_harabasz": calinski,

            "min_cluster_size": min_cluster,

            "max_cluster_size": max_cluster,

            "min_cluster_pct": min_cluster_pct,

            "time_seconds": tiempo
        }


        resultados_region.append(
            resultado
        )

        todos_resultados.append(
            resultado
        )


    # ========================================================
    # GUARDAR RESULTADOS DE ESTA REGIÓN
    # ========================================================

    region_df = pd.DataFrame(
        resultados_region
    )

    region_df.to_csv(
        CLUSTER_DIR /
        f"{region.lower()}_k_evaluation.csv",
        index=False
    )


    # ========================================================
    # MEJORES K SEGÚN CADA MÉTRICA
    # ========================================================

    best_silhouette = (
        region_df
        .loc[
            region_df["silhouette"].idxmax()
        ]
    )

    best_davies = (
        region_df
        .loc[
            region_df["davies_bouldin"].idxmin()
        ]
    )

    best_calinski = (
        region_df
        .loc[
            region_df["calinski_harabasz"].idxmax()
        ]
    )


    print("\nMejores resultados:")

    print(
        f"Silhouette       → "
        f"K={int(best_silhouette['k'])} | "
        f"{best_silhouette['silhouette']:.4f}"
    )

    print(
        f"Davies-Bouldin   → "
        f"K={int(best_davies['k'])} | "
        f"{best_davies['davies_bouldin']:.4f}"
    )

    print(
        f"Calinski-Harabasz→ "
        f"K={int(best_calinski['k'])} | "
        f"{best_calinski['calinski_harabasz']:.2f}"
    )


# ============================================================
# RESULTADOS GLOBALES
# ============================================================

results_df = pd.DataFrame(
    todos_resultados
)

results_df.to_csv(
    CLUSTER_DIR /
    "all_regions_k_evaluation.csv",
    index=False
)


# ============================================================
# RESUMEN
# ============================================================

resumen = []

for region in regiones:

    r = results_df[
        results_df["region"] == region
    ]

    sil = r.loc[
        r["silhouette"].idxmax()
    ]

    db = r.loc[
        r["davies_bouldin"].idxmin()
    ]

    ch = r.loc[
        r["calinski_harabasz"].idxmax()
    ]

    resumen.append({
        "region": region,

        "best_k_silhouette":
            int(sil["k"]),

        "best_silhouette":
            sil["silhouette"],

        "best_k_davies":
            int(db["k"]),

        "best_davies":
            db["davies_bouldin"],

        "best_k_calinski":
            int(ch["k"]),

        "best_calinski":
            ch["calinski_harabasz"]
    })


summary_df = pd.DataFrame(
    resumen
)

summary_df.to_csv(
    CLUSTER_DIR /
    "k_evaluation_summary.csv",
    index=False
)


print("\n" + "=" * 75)
print("RESUMEN GENERAL")
print("=" * 75)

display(summary_df)


print(
    f"\nTiempo total: "
    f"{(time.time() - inicio_total) / 60:.2f} minutos"
)

print(
    f"\nResultados guardados en:\n"
    f"{CLUSTER_DIR}"
)

print(
    "\n✅ Evaluación de K terminada."
)


XR_ELBOW
Imágenes: 2,006 | Dimensiones PCA: 372


Evaluando XR_ELBOW:   0%|          | 0/29 [00:00<?, ?it/s]


Mejores resultados:
Silhouette       → K=2 | 0.1328
Davies-Bouldin   → K=2 | 2.6568
Calinski-Harabasz→ K=2 | 267.64

XR_FINGER
Imágenes: 1,968 | Dimensiones PCA: 362


Evaluando XR_FINGER:   0%|          | 0/29 [00:00<?, ?it/s]


Mejores resultados:
Silhouette       → K=2 | 0.1446
Davies-Bouldin   → K=2 | 2.2574
Calinski-Harabasz→ K=2 | 322.82

XR_FOREARM
Imágenes: 661 | Dimensiones PCA: 261


...


Mejores resultados:
Silhouette       → K=2 | 0.1136
Davies-Bouldin   → K=7 | 2.8301
Calinski-Harabasz→ K=2 | 403.73

XR_WRIST
Imágenes: 3,987 | Dimensiones PCA: 412


Evaluando XR_WRIST:   0%|          | 0/29 [00:00<?, ?it/s]


Mejores resultados:
Silhouette       → K=2 | 0.0923
Davies-Bouldin   → K=7 | 2.7690
Calinski-Harabasz→ K=2 | 395.52

RESUMEN GENERAL


,region,best_k_silhouette,best_silhouette,best_k_davies,best_davies,best_k_calinski,best_calinski
0,XR_ELBOW,2,0.132824,2,2.656806,2,267.635681
1,XR_FINGER,2,0.144558,2,2.257394,2,322.816589
2,XR_FOREARM,2,0.122490,4,2.385238,2,73.183975
3,XR_HAND,2,0.102153,3,2.771451,2,159.620621
4,XR_HUMERUS,3,0.077480,28,2.623739,2,49.073475
5,XR_SHOULDER,2,0.113560,7,2.830057,2,403.732605
6,XR_WRIST,2,0.092261,7,2.769020,2,395.521423



Tiempo total: 4.64 minutos

Resultados guardados en:
/content/drive/MyDrive/MURA_TT/clustering

✅ Evaluación de K terminada.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

CLUSTER_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/clustering"
)

RESULTS_FILE = (
    CLUSTER_DIR /
    "all_regions_k_evaluation.csv"
)

results_df = pd.read_csv(RESULTS_FILE)

regiones = [
    "XR_ELBOW",
    "XR_FINGER",
    "XR_FOREARM",
    "XR_HAND",
    "XR_HUMERUS",
    "XR_SHOULDER",
    "XR_WRIST"
]

# ============================================================
# MOSTRAR K SELECCIONADOS
# Para analizar tendencia sin imprimir 203 filas
# ============================================================

K_MOSTRAR = [
    2, 3, 4, 5,
    7, 10, 12, 15,
    20, 25, 30
]


for region in regiones:

    print("\n" + "=" * 95)
    print(region)
    print("=" * 95)

    r = (
        results_df[
            (results_df["region"] == region) &
            (results_df["k"].isin(K_MOSTRAR))
        ]
        [
            [
                "k",
                "silhouette",
                "davies_bouldin",
                "calinski_harabasz",
                "min_cluster_size",
                "max_cluster_size",
                "min_cluster_pct"
            ]
        ]
        .copy()
    )

    # Redondear solo para visualización
    r["silhouette"] = r["silhouette"].round(4)
    r["davies_bouldin"] = r["davies_bouldin"].round(4)
    r["calinski_harabasz"] = (
        r["calinski_harabasz"].round(2)
    )
    r["min_cluster_pct"] = (
        r["min_cluster_pct"].round(2)
    )

    display(r.reset_index(drop=True))


XR_ELBOW


,k,silhouette,davies_bouldin,calinski_harabasz,min_cluster_size,max_cluster_size,min_cluster_pct
0,2,0.1328,2.6568,267.64,795,1211,39.63
1,3,0.0849,3.0966,191.36,644,686,32.10
2,4,0.0976,2.6948,157.97,45,687,2.24
3,5,0.0848,2.9042,139.76,45,598,2.24
4,7,0.0774,2.9670,116.11,46,508,2.29
5,10,0.0684,3.0257,90.56,45,426,2.24
6,12,0.0436,3.0926,79.66,45,306,2.24
7,15,0.0396,3.1234,68.44,45,194,2.24
8,20,0.0435,3.3189,54.99,21,245,1.05
9,25,0.0324,3.2639,46.41,26,158,1.30



XR_FINGER


,k,silhouette,davies_bouldin,calinski_harabasz,min_cluster_size,max_cluster_size,min_cluster_pct
0,2,0.1446,2.2574,322.82,648,1320,32.93
1,3,0.1158,2.6016,250.93,470,862,23.88
2,4,0.1095,3.1327,193.14,396,634,20.12
3,5,0.0888,3.0680,160.29,272,633,13.82
4,7,0.0624,3.2633,123.22,165,382,8.38
5,10,0.0481,3.2857,93.85,128,281,6.50
6,12,0.0538,3.0580,82.32,49,276,2.49
7,15,0.0382,3.2783,69.24,58,228,2.95
8,20,0.0385,3.1344,56.74,42,165,2.13
9,25,0.0346,3.2819,48.09,24,151,1.22



XR_FOREARM


...

,k,silhouette,davies_bouldin,calinski_harabasz,min_cluster_size,max_cluster_size,min_cluster_pct
0,2,0.0741,3.3445,49.07,252,347,42.07
1,3,0.0775,2.6713,40.91,20,339,3.34
2,4,0.0590,3.1665,36.03,20,220,3.34
3,5,0.0570,3.1509,32.23,20,203,3.34
4,7,0.0464,3.0716,27.06,20,129,3.34
5,10,0.0486,3.1187,21.78,18,102,3.01
6,12,0.0449,3.0020,19.69,20,91,3.34
7,15,0.0359,2.8873,16.57,6,104,1.00
8,20,0.0358,2.8980,14.03,6,58,1.00
9,25,0.0428,2.6951,12.23,5,70,0.83



XR_SHOULDER


,k,silhouette,davies_bouldin,calinski_harabasz,min_cluster_size,max_cluster_size,min_cluster_pct
0,2,0.1136,3.1507,403.73,1631,2537,39.13
1,3,0.1119,3.1049,304.94,781,2540,18.74
2,4,0.0632,3.1096,271.55,343,1582,8.23
3,5,0.0733,3.0103,252.36,335,1555,8.04
4,7,0.0570,2.8301,205.68,52,1051,1.25
5,10,0.0510,3.0683,163.88,51,719,1.22
6,12,0.0509,3.0715,144.30,52,634,1.25
7,15,0.0469,3.3909,122.50,51,597,1.22
8,20,0.0385,3.4131,99.04,51,386,1.22
9,25,0.0398,3.4013,84.08,51,329,1.22



XR_WRIST


,k,silhouette,davies_bouldin,calinski_harabasz,min_cluster_size,max_cluster_size,min_cluster_pct
0,2,0.0923,3.1063,395.52,1877,2110,47.08
1,3,0.0778,3.3549,305.18,1275,1377,31.98
2,4,0.0671,3.2722,252.86,533,1277,13.37
3,5,0.0725,2.8292,226.28,34,1285,0.85
4,7,0.0721,2.7690,190.91,34,1159,0.85
5,10,0.0537,2.9883,156.28,34,680,0.85
6,12,0.0448,2.9770,138.80,34,518,0.85
7,15,0.0391,3.1388,118.55,34,461,0.85
8,20,0.0392,3.1201,97.07,34,390,0.85
9,25,0.0366,3.0650,83.13,34,288,0.85


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

from itertools import combinations
from tqdm.auto import tqdm


# ============================================================
# CONFIGURACIÓN
# ============================================================

PCA_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/pca"
)

CLUSTER_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/clustering"
)

K_VALUES = range(2, 16)

SEEDS = [
    11,
    22,
    33,
    44,
    55
]


regiones = [
    "XR_ELBOW",
    "XR_FINGER",
    "XR_FOREARM",
    "XR_HAND",
    "XR_HUMERUS",
    "XR_SHOULDER",
    "XR_WRIST"
]


# ============================================================
# RESULTADOS
# ============================================================

resultados = []


# ============================================================
# EVALUAR REGIONES
# ============================================================

for region in regiones:

    print("\n" + "=" * 75)
    print(region)
    print("=" * 75)

    archivo = (
        PCA_DIR /
        f"{region.lower()}_pca.npz"
    )

    datos = np.load(archivo)

    X = datos["X_pca"]


    for k in tqdm(
        K_VALUES,
        desc=f"Estabilidad {region}"
    ):

        particiones = []


        # ----------------------------------------------------
        # EJECUTAR K-MEANS CON DIFERENTES SEMILLAS
        # ----------------------------------------------------

        for seed in SEEDS:

            kmeans = KMeans(
                n_clusters=k,
                init="k-means++",
                n_init=10,
                max_iter=300,
                random_state=seed
            )

            labels = kmeans.fit_predict(X)

            particiones.append(labels)


        # ----------------------------------------------------
        # ARI ENTRE TODOS LOS PARES DE EJECUCIONES
        #
        # 5 semillas → 10 comparaciones
        # ----------------------------------------------------

        ari_scores = []

        for i, j in combinations(
            range(len(particiones)),
            2
        ):

            ari = adjusted_rand_score(
                particiones[i],
                particiones[j]
            )

            ari_scores.append(ari)


        # ----------------------------------------------------
        # RESULTADOS
        # ----------------------------------------------------

        resultados.append({

            "region": region,

            "k": k,

            "ari_mean":
                np.mean(ari_scores),

            "ari_std":
                np.std(ari_scores),

            "ari_min":
                np.min(ari_scores),

            "ari_max":
                np.max(ari_scores)
        })


# ============================================================
# DATAFRAME
# ============================================================

stability_df = pd.DataFrame(
    resultados
)


# ============================================================
# GUARDAR
# ============================================================

STABILITY_FILE = (
    CLUSTER_DIR /
    "kmeans_stability.csv"
)

stability_df.to_csv(
    STABILITY_FILE,
    index=False
)


# ============================================================
# MOSTRAR RESULTADOS
# ============================================================

for region in regiones:

    print("\n" + "=" * 90)
    print(region)
    print("=" * 90)

    r = (
        stability_df[
            stability_df["region"] == region
        ]
        .copy()
    )

    for col in [
        "ari_mean",
        "ari_std",
        "ari_min",
        "ari_max"
    ]:

        r[col] = r[col].round(4)

    display(
        r.reset_index(drop=True)
    )


print(
    f"\nResultados guardados en:\n"
    f"{STABILITY_FILE}"
)

print(
    "\n✅ Evaluación de estabilidad terminada."
)


XR_ELBOW


Estabilidad XR_ELBOW:   0%|          | 0/14 [00:00<?, ?it/s]


XR_FINGER


Estabilidad XR_FINGER:   0%|          | 0/14 [00:00<?, ?it/s]


XR_FOREARM


...


XR_SHOULDER


,region,k,ari_mean,ari_std,ari_min,ari_max
0,XR_SHOULDER,2,1.0000,0.0000,1.0000,1.0000
1,XR_SHOULDER,3,1.0000,0.0000,1.0000,1.0000
2,XR_SHOULDER,4,0.9979,0.0014,0.9958,1.0000
3,XR_SHOULDER,5,0.9862,0.0086,0.9730,0.9963
4,XR_SHOULDER,6,0.9863,0.0059,0.9761,0.9945
5,XR_SHOULDER,7,0.9405,0.0449,0.8843,0.9871
6,XR_SHOULDER,8,0.7940,0.1243,0.6896,0.9864
7,XR_SHOULDER,9,0.8601,0.1481,0.6756,0.9900
8,XR_SHOULDER,10,0.8440,0.0671,0.7769,0.9607
9,XR_SHOULDER,11,0.8084,0.1088,0.7199,0.9767



XR_WRIST


,region,k,ari_mean,ari_std,ari_min,ari_max
0,XR_WRIST,2,0.9940,0.0053,0.9850,1.0000
1,XR_WRIST,3,0.9956,0.0039,0.9904,1.0000
2,XR_WRIST,4,0.9824,0.0078,0.9689,0.9946
3,XR_WRIST,5,0.8692,0.0989,0.7029,0.9932
4,XR_WRIST,6,0.8206,0.1349,0.6432,0.9927
5,XR_WRIST,7,0.8108,0.1603,0.5824,0.9887
6,XR_WRIST,8,0.9738,0.0101,0.9589,0.9891
7,XR_WRIST,9,0.9805,0.0087,0.9686,0.9937
8,XR_WRIST,10,0.8098,0.1228,0.7025,0.9926
9,XR_WRIST,11,0.9658,0.0137,0.9429,0.9868



Resultados guardados en:
/content/drive/MyDrive/MURA_TT/clustering/kmeans_stability.csv

✅ Evaluación de estabilidad terminada.


In [ ]:
from pathlib import Path
from PIL import Image

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans


# ============================================================
# CONFIGURACIÓN
# ============================================================

PCA_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/pca"
)

EMBEDDINGS_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/embeddings"
)

VIS_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/cluster_visualization"
)

VIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# K provisional por región
K_PROVISIONAL = {
    "XR_ELBOW": 6,
    "XR_FINGER": 6,
    "XR_FOREARM": 6,
    "XR_HAND": 6,
    "XR_HUMERUS": 4,
    "XR_SHOULDER": 6,
    "XR_WRIST": 9
}


N_IMAGES_PER_CLUSTER = 6
RANDOM_STATE = 42


# ============================================================
# MANIFEST
# ============================================================

manifest = pd.read_csv(
    EMBEDDINGS_DIR /
    "densenet121_imagenet_abnormal_train_manifest.csv"
)


# ============================================================
# FUNCIÓN PARA MOSTRAR CLUSTERS
# ============================================================

def crear_mosaico_region(
    region,
    k,
    n_images=6
):

    # --------------------------------------------------------
    # CARGAR PCA
    # --------------------------------------------------------

    data = np.load(
        PCA_DIR /
        f"{region.lower()}_pca.npz"
    )

    X = data["X_pca"]

    global_indices = data["indices"]


    # --------------------------------------------------------
    # K-MEANS
    # --------------------------------------------------------

    kmeans = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        max_iter=300,
        random_state=RANDOM_STATE
    )

    labels = kmeans.fit_predict(X)


    # --------------------------------------------------------
    # DISTANCIA AL CENTROIDE
    # --------------------------------------------------------

    distances = np.linalg.norm(
        X - kmeans.cluster_centers_[labels],
        axis=1
    )


    # --------------------------------------------------------
    # FIGURA
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        k,
        n_images,
        figsize=(3 * n_images, 3 * k)
    )

    if k == 1:
        axes = np.array([axes])


    for cluster_id in range(k):

        cluster_positions = np.where(
            labels == cluster_id
        )[0]

        # Ordenar por distancia al centroide.
        # Aquí queremos imágenes REPRESENTATIVAS.
        ordered = cluster_positions[
            np.argsort(
                distances[cluster_positions]
            )
        ]

        # Evitar varias imágenes del mismo estudio
        selected = []
        used_studies = set()

        for pos in ordered:

            global_idx = global_indices[pos]

            study_id = manifest.iloc[
                global_idx
            ]["study_id"]

            if study_id in used_studies:
                continue

            selected.append(pos)
            used_studies.add(study_id)

            if len(selected) == n_images:
                break


        cluster_size = len(
            cluster_positions
        )


        for col in range(n_images):

            ax = axes[
                cluster_id,
                col
            ]

            ax.axis("off")


            if col >= len(selected):
                continue


            pos = selected[col]

            global_idx = global_indices[pos]

            row = manifest.iloc[
                global_idx
            ]

            path = row["local_path"]


            with Image.open(path) as img:

                img = img.convert("L")

                ax.imshow(
                    img,
                    cmap="gray"
                )


            if col == 0:

                ax.set_title(
                    f"Cluster {cluster_id}\n"
                    f"n={cluster_size}",
                    fontsize=10
                )

            else:

                ax.set_title(
                    row["image_name"],
                    fontsize=8
                )


    fig.suptitle(
        f"{region} — K={k} "
        f"— imágenes cercanas al centroide",
        fontsize=16
    )

    plt.tight_layout()

    output_file = (
        VIS_DIR /
        f"{region.lower()}_k{k}_representatives.png"
    )

    plt.savefig(
        output_file,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

    plt.close()

    print(
        f"Guardado: {output_file}"
    )


# ============================================================
# GENERAR MOSAICOS
# ============================================================

for region, k in K_PROVISIONAL.items():

    print("\n" + "=" * 70)
    print(
        f"{region} | K={k}"
    )
    print("=" * 70)

    crear_mosaico_region(
        region=region,
        k=k,
        n_images=N_IMAGES_PER_CLUSTER
    )

In [ ]:
import shutil

CARPETA = "/content/drive/MyDrive/MURA_TT/cluster_visualization"
SALIDA = "/content/drive/MyDrive/MURA_TT/mosaicos_clusters"

shutil.make_archive(
    SALIDA,
    "zip",
    CARPETA
)

print("ZIP creado:")
print(SALIDA + ".zip")

ZIP creado:
/content/drive/MyDrive/MURA_TT/mosaicos_clusters.zip


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans


# ============================================================
# CONFIGURACIÓN
# ============================================================

PCA_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/pca"
)

CLUSTER_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/clustering"
)

CLUSTER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

N_TOTAL = 2000

RANDOM_STATE = 42


K_FINAL = {
    "XR_ELBOW": 6,
    "XR_FINGER": 6,
    "XR_FOREARM": 6,
    "XR_HAND": 6,
    "XR_HUMERUS": 4,
    "XR_SHOULDER": 6,
    "XR_WRIST": 9
}


# ============================================================
# FUNCIÓN DE REDONDEO QUE CONSERVA EL TOTAL
# ============================================================

def largest_remainder(values, total):

    values = np.asarray(
        values,
        dtype=float
    )

    raw = (
        values /
        values.sum() *
        total
    )

    base = np.floor(raw).astype(int)

    remainder = (
        total - base.sum()
    )

    if remainder > 0:

        order = np.argsort(
            -(raw - base)
        )

        base[
            order[:remainder]
        ] += 1

    return base


# ============================================================
# 1. EJECUTAR CLUSTERING FINAL
# ============================================================

cluster_rows = []

labels_por_region = {}


for region, k in K_FINAL.items():

    archivo = (
        PCA_DIR /
        f"{region.lower()}_pca.npz"
    )

    data = np.load(archivo)

    X = data["X_pca"]
    indices = data["indices"]


    kmeans = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        max_iter=300,
        random_state=RANDOM_STATE
    )

    labels = kmeans.fit_predict(X)


    labels_por_region[region] = {
        "labels": labels,
        "indices": indices,
        "centers": kmeans.cluster_centers_
    }


    sizes = np.bincount(
        labels,
        minlength=k
    )


    for cluster_id, size in enumerate(sizes):

        cluster_rows.append({
            "region": region,
            "cluster": cluster_id,
            "cluster_size": int(size)
        })


cluster_sizes = pd.DataFrame(
    cluster_rows
)


# ============================================================
# 2. TAMAÑOS POR REGIÓN
# ============================================================

region_sizes = (
    cluster_sizes
    .groupby("region", as_index=False)
    ["cluster_size"]
    .sum()
    .rename(
        columns={
            "cluster_size":
            "region_size"
        }
    )
)


# ============================================================
# 3. ESTRATEGIA A:
# TOTALMENTE PROPORCIONAL
# ============================================================

region_sizes["allocation_proportional"] = (
    largest_remainder(
        region_sizes["region_size"],
        N_TOTAL
    )
)


# ============================================================
# 4. ESTRATEGIA B:
# IGUAL POR REGIÓN
# ============================================================

region_sizes["allocation_equal"] = (
    largest_remainder(
        np.ones(
            len(region_sizes)
        ),
        N_TOTAL
    )
)


# ============================================================
# 5. ESTRATEGIA C:
# HÍBRIDA 50/50
#
# 50 % del presupuesto:
#     igual entre regiones
#
# 50 %:
#     proporcional al tamaño
# ============================================================

equal_component = (
    np.ones(
        len(region_sizes)
    ) /
    len(region_sizes)
)

proportional_component = (
    region_sizes["region_size"]
    .to_numpy() /
    region_sizes["region_size"].sum()
)

hybrid_weights = (
    0.5 * equal_component +
    0.5 * proportional_component
)


region_sizes["allocation_hybrid"] = (
    largest_remainder(
        hybrid_weights,
        N_TOTAL
    )
)


# ============================================================
# 6. PORCENTAJES
# ============================================================

region_sizes[
    "dataset_pct"
] = (
    region_sizes["region_size"] /
    region_sizes["region_size"].sum()
    * 100
)

for col in [
    "allocation_proportional",
    "allocation_equal",
    "allocation_hybrid"
]:

    region_sizes[
        col + "_pct"
    ] = (
        region_sizes[col] /
        N_TOTAL *
        100
    )


# ============================================================
# 7. MOSTRAR COMPARACIÓN REGIONAL
# ============================================================

print("=" * 90)
print("COMPARACIÓN DE ASIGNACIÓN POR REGIÓN")
print("=" * 90)

display(
    region_sizes[
        [
            "region",
            "region_size",
            "dataset_pct",
            "allocation_proportional",
            "allocation_equal",
            "allocation_hybrid"
        ]
    ]
    .sort_values("region")
    .reset_index(drop=True)
)


print("\nTotales:")

print(
    "Proporcional:",
    region_sizes[
        "allocation_proportional"
    ].sum()
)

print(
    "Igual:",
    region_sizes[
        "allocation_equal"
    ].sum()
)

print(
    "Híbrida:",
    region_sizes[
        "allocation_hybrid"
    ].sum()
)


# ============================================================
# 8. DISTRIBUCIÓN HÍBRIDA DENTRO DE CADA CLUSTER
#
# Damos inicialmente una cobertura base pequeña:
#
# min(10, tamaño del cluster)
#
# y distribuimos el resto proporcionalmente entre clusters.
#
# Esto NO es todavía la selección final.
# ============================================================

cluster_allocations = []


for _, region_row in region_sizes.iterrows():

    region = region_row["region"]

    region_quota = int(
        region_row[
            "allocation_hybrid"
        ]
    )

    r = (
        cluster_sizes[
            cluster_sizes["region"] == region
        ]
        .copy()
        .sort_values("cluster")
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # COBERTURA BASE
    # --------------------------------------------------------

    r["base"] = np.minimum(
        10,
        r["cluster_size"]
    )


    base_total = int(
        r["base"].sum()
    )

    remaining = (
        region_quota -
        base_total
    )


    # --------------------------------------------------------
    # CAPACIDAD DISPONIBLE DESPUÉS DE LA BASE
    # --------------------------------------------------------

    capacity = (
        r["cluster_size"] -
        r["base"]
    ).to_numpy()


    if remaining > 0:

        extra = largest_remainder(
            capacity,
            remaining
        )

        # Seguridad: nunca superar tamaño real
        extra = np.minimum(
            extra,
            capacity
        )

    else:

        extra = np.zeros(
            len(r),
            dtype=int
        )


    r["allocation"] = (
        r["base"] +
        extra
    )


    # --------------------------------------------------------
    # CORRECCIÓN EN CASO DE REDONDEO/CAPACIDAD
    # --------------------------------------------------------

    deficit = (
        region_quota -
        r["allocation"].sum()
    )


    while deficit > 0:

        available = (
            r["cluster_size"] -
            r["allocation"]
        )

        idx = available.idxmax()

        if available.loc[idx] <= 0:
            break

        r.loc[
            idx,
            "allocation"
        ] += 1

        deficit -= 1


    r["region_quota"] = (
        region_quota
    )


    r["cluster_dataset_pct"] = (
        r["cluster_size"] /
        r["cluster_size"].sum()
        * 100
    )


    r["cluster_sample_pct"] = (
        r["allocation"] /
        region_quota
        * 100
    )


    cluster_allocations.append(
        r
    )


cluster_allocation_df = pd.concat(
    cluster_allocations,
    ignore_index=True
)


# ============================================================
# 9. MOSTRAR ASIGNACIÓN POR CLUSTER
# ============================================================

print("\n" + "=" * 90)
print("PROPUESTA HÍBRIDA POR CLUSTER")
print("=" * 90)


for region in K_FINAL:

    print("\n" + "-" * 70)
    print(region)
    print("-" * 70)

    r = (
        cluster_allocation_df[
            cluster_allocation_df[
                "region"
            ] == region
        ]
        [
            [
                "cluster",
                "cluster_size",
                "cluster_dataset_pct",
                "allocation",
                "cluster_sample_pct"
            ]
        ]
        .copy()
    )

    r[
        "cluster_dataset_pct"
    ] = (
        r[
            "cluster_dataset_pct"
        ].round(2)
    )

    r[
        "cluster_sample_pct"
    ] = (
        r[
            "cluster_sample_pct"
        ].round(2)
    )

    display(
        r.reset_index(drop=True)
    )


# ============================================================
# 10. VALIDACIONES
# ============================================================

print("\n" + "=" * 90)
print("VALIDACIÓN")
print("=" * 90)

print(
    "\nTotal asignado:",
    cluster_allocation_df[
        "allocation"
    ].sum()
)

print(
    "Objetivo:",
    N_TOTAL
)


excess = (
    cluster_allocation_df[
        "allocation"
    ] >
    cluster_allocation_df[
        "cluster_size"
    ]
).sum()


print(
    "Clusters con asignación "
    "mayor que su tamaño:",
    excess
)


# ============================================================
# 11. GUARDAR
# ============================================================

region_sizes.to_csv(
    CLUSTER_DIR /
    "sampling_region_strategies.csv",
    index=False
)

cluster_allocation_df.to_csv(
    CLUSTER_DIR /
    "sampling_hybrid_cluster_proposal.csv",
    index=False
)


print(
    "\n✅ Comparación de estrategias terminada."
)

COMPARACIÓN DE ASIGNACIÓN POR REGIÓN


,region,region_size,dataset_pct,allocation_proportional,allocation_equal,allocation_hybrid
0,XR_ELBOW,2006,13.487528,270,286,278
1,XR_FINGER,1968,13.232031,265,286,275
2,XR_FOREARM,661,4.444295,89,286,187
3,XR_HAND,1484,9.977812,200,286,243
4,XR_HUMERUS,599,4.027432,80,286,183
5,XR_SHOULDER,4168,28.023936,560,285,423
6,XR_WRIST,3987,26.806966,536,285,411



Totales:
Proporcional: 2000
Igual: 2000
Híbrida: 2000

PROPUESTA HÍBRIDA POR CLUSTER

----------------------------------------------------------------------
XR_ELBOW
----------------------------------------------------------------------


,cluster,cluster_size,cluster_dataset_pct,allocation,cluster_sample_pct
0,0,441,21.98,58,20.86
1,1,514,25.62,67,24.10
2,2,289,14.41,41,14.75
3,3,45,2.24,14,5.04
4,4,376,18.74,51,18.35
5,5,341,17.00,47,16.91



----------------------------------------------------------------------
XR_FINGER
----------------------------------------------------------------------


...


----------------------------------------------------------------------
XR_SHOULDER
----------------------------------------------------------------------


,cluster,cluster_size,cluster_dataset_pct,allocation,cluster_sample_pct
0,0,698,16.75,71,16.78
1,1,335,8.04,39,9.22
2,2,826,19.82,82,19.39
3,3,771,18.50,77,18.20
4,4,483,11.59,52,12.29
5,5,1055,25.31,102,24.11



----------------------------------------------------------------------
XR_WRIST
----------------------------------------------------------------------


,cluster,cluster_size,cluster_dataset_pct,allocation,cluster_sample_pct
0,0,34,0.85,12,2.92
1,1,527,13.22,52,12.65
2,2,459,11.51,47,11.44
3,3,522,13.09,52,12.65
4,4,347,8.70,38,9.25
5,5,605,15.17,59,14.36
6,6,370,9.28,40,9.73
7,7,145,3.64,21,5.11
8,8,978,24.53,90,21.90



VALIDACIÓN

Total asignado: 2000
Objetivo: 2000
Clusters con asignación mayor que su tamaño: 0

✅ Comparación de estrategias terminada.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans


# ============================================================
# CONFIGURACIÓN
# ============================================================

PCA_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/pca"
)

EMBEDDINGS_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/embeddings"
)

CLUSTER_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/clustering"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/MURA_TT/annotation_sample"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RANDOM_STATE = 42


K_FINAL = {
    "XR_ELBOW": 6,
    "XR_FINGER": 6,
    "XR_FOREARM": 6,
    "XR_HAND": 6,
    "XR_HUMERUS": 4,
    "XR_SHOULDER": 6,
    "XR_WRIST": 9
}


# ============================================================
# CARGAR MANIFEST Y ASIGNACIONES
# ============================================================

manifest = pd.read_csv(
    EMBEDDINGS_DIR /
    "densenet121_imagenet_abnormal_train_manifest.csv"
)

allocation_df = pd.read_csv(
    CLUSTER_DIR /
    "sampling_hybrid_cluster_proposal.csv"
)


# ============================================================
# FUNCIÓN PARA REPARTIR CUOTA 60 / 20 / 20
# ============================================================

def split_quota(n):

    raw = np.array([
        0.60,
        0.20,
        0.20
    ]) * n

    base = np.floor(raw).astype(int)

    remainder = n - base.sum()

    order = np.argsort(
        -(raw - base)
    )

    base[
        order[:remainder]
    ] += 1

    return {
        "central": int(base[0]),
        "intermediate": int(base[1]),
        "peripheral": int(base[2])
    }


# ============================================================
# FUNCIÓN DE SELECCIÓN
# ============================================================

def select_cases(
    candidates,
    quota,
    used_studies_global
):

    if quota <= 0:
        return candidates.iloc[0:0].copy()


    candidates = candidates.copy()


    # --------------------------------------------------------
    # Aleatoriedad reproducible dentro del estrato
    # --------------------------------------------------------

    candidates = candidates.sample(
        frac=1,
        random_state=RANDOM_STATE
    )


    selected_indices = []


    # --------------------------------------------------------
    # PRIMERA PASADA:
    # estudios que todavía NO han sido seleccionados
    # --------------------------------------------------------

    for idx, row in candidates.iterrows():

        study = row["study_id"]

        if study not in used_studies_global:

            selected_indices.append(idx)

            used_studies_global.add(study)

        if len(selected_indices) == quota:
            break


    # --------------------------------------------------------
    # SEGUNDA PASADA:
    # si no alcanzó, permitir segunda imagen del mismo estudio
    # --------------------------------------------------------

    if len(selected_indices) < quota:

        already_selected = set(
            selected_indices
        )

        for idx, row in candidates.iterrows():

            if idx in already_selected:
                continue

            selected_indices.append(idx)

            if len(selected_indices) == quota:
                break


    return candidates.loc[
        selected_indices
    ].copy()


# ============================================================
# CONSTRUIR TABLA COMPLETA CON CLUSTERS Y DISTANCIAS
# ============================================================

all_clustered = []


for region, k in K_FINAL.items():

    print(
        f"Procesando {region} | K={k}"
    )


    data = np.load(
        PCA_DIR /
        f"{region.lower()}_pca.npz"
    )

    X = data["X_pca"]

    global_indices = data["indices"]


    kmeans = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        max_iter=300,
        random_state=RANDOM_STATE
    )

    labels = kmeans.fit_predict(X)


    distances = np.linalg.norm(
        X -
        kmeans.cluster_centers_[labels],
        axis=1
    )


    region_df = (
        manifest
        .iloc[global_indices]
        .copy()
    )

    region_df[
        "global_embedding_index"
    ] = global_indices

    region_df[
        "cluster"
    ] = labels

    region_df[
        "distance_to_centroid"
    ] = distances


    all_clustered.append(
        region_df
    )


clustered_df = pd.concat(
    all_clustered,
    ignore_index=True
)


# ============================================================
# ASIGNAR ZONA DE DISTANCIA DENTRO DE CADA CLUSTER
# ============================================================

clustered_df[
    "distance_percentile"
] = np.nan

clustered_df[
    "sampling_zone"
] = None


for (region, cluster), idx in (
    clustered_df
    .groupby(
        ["region", "cluster"]
    )
    .groups
    .items()
):

    idx = list(idx)

    distances = (
        clustered_df.loc[
            idx,
            "distance_to_centroid"
        ]
    )


    ranks = (
        distances.rank(
            method="average",
            pct=True
        )
    )


    clustered_df.loc[
        idx,
        "distance_percentile"
    ] = ranks


    # --------------------------------------------------------
    # ZONAS
    # --------------------------------------------------------

    clustered_df.loc[
        idx,
        "sampling_zone"
    ] = np.select(

        [
            ranks <= 0.50,

            (
                (ranks > 0.50) &
                (ranks <= 0.80)
            ),

            (
                (ranks > 0.80) &
                (ranks <= 0.95)
            )
        ],

        [
            "central",
            "intermediate",
            "peripheral"
        ],

        default="extreme"
    )


# ============================================================
# SELECCIÓN
# ============================================================

selected_parts = []

used_studies_global = set()


for _, allocation_row in (
    allocation_df
    .sort_values(
        ["region", "cluster"]
    )
    .iterrows()
):

    region = allocation_row[
        "region"
    ]

    cluster = int(
        allocation_row[
            "cluster"
        ]
    )

    quota = int(
        allocation_row[
            "allocation"
        ]
    )


    cluster_cases = (
        clustered_df[
            (clustered_df["region"] == region) &
            (clustered_df["cluster"] == cluster)
        ]
        .copy()
    )


    zone_quota = split_quota(
        quota
    )


    cluster_selected = []


    for zone in [
        "central",
        "intermediate",
        "peripheral"
    ]:

        candidates = (
            cluster_cases[
                cluster_cases[
                    "sampling_zone"
                ] == zone
            ]
            .copy()
        )


        selected = select_cases(
            candidates,
            zone_quota[zone],
            used_studies_global
        )


        selected[
            "selection_zone"
        ] = zone


        cluster_selected.append(
            selected
        )


    selected_cluster_df = pd.concat(
        cluster_selected,
        ignore_index=False
    )


    # --------------------------------------------------------
    # SI POR ALGUNA RAZÓN NO ALCANZÓ LA CUOTA:
    # rellenar desde casos no seleccionados, excluyendo
    # inicialmente los extremos si es posible.
    # --------------------------------------------------------

    deficit = (
        quota -
        len(selected_cluster_df)
    )


    if deficit > 0:

        selected_ids = set(
            selected_cluster_df[
                "global_embedding_index"
            ]
        )


        backup = cluster_cases[
            ~cluster_cases[
                "global_embedding_index"
            ].isin(selected_ids)
            &
            (
                cluster_cases[
                    "sampling_zone"
                ] != "extreme"
            )
        ].copy()


        extra = select_cases(
            backup,
            deficit,
            used_studies_global
        )


        extra[
            "selection_zone"
        ] = "backup"


        selected_cluster_df = pd.concat(
            [
                selected_cluster_df,
                extra
            ],
            ignore_index=False
        )


    # --------------------------------------------------------
    # ÚLTIMO RECURSO:
    # usar extremos solo si fuera necesario
    # --------------------------------------------------------

    deficit = (
        quota -
        len(selected_cluster_df)
    )


    if deficit > 0:

        selected_ids = set(
            selected_cluster_df[
                "global_embedding_index"
            ]
        )


        extreme_backup = cluster_cases[
            ~cluster_cases[
                "global_embedding_index"
            ].isin(selected_ids)
        ].copy()


        extra = select_cases(
            extreme_backup,
            deficit,
            used_studies_global
        )


        extra[
            "selection_zone"
        ] = "extreme_backup"


        selected_cluster_df = pd.concat(
            [
                selected_cluster_df,
                extra
            ],
            ignore_index=False
        )


    selected_parts.append(
        selected_cluster_df
    )


# ============================================================
# MUESTRA FINAL
# ============================================================

sample_df = pd.concat(
    selected_parts,
    ignore_index=True
)


# ============================================================
# CREAR ID CIEGO PARA ANOTACIÓN
# ============================================================

sample_df = sample_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)


sample_df[
    "annotation_case_id"
] = [
    f"MURA_TT_{i:04d}"
    for i in range(
        1,
        len(sample_df) + 1
    )
]


# ============================================================
# VALIDACIONES GENERALES
# ============================================================

print("\n" + "=" * 80)
print("VALIDACIÓN DE LA MUESTRA")
print("=" * 80)


print(
    "\nTotal seleccionado:",
    len(sample_df)
)


print(
    "Imágenes únicas:",
    sample_df[
        "global_embedding_index"
    ].nunique()
)


print(
    "Estudios únicos:",
    sample_df[
        "study_id"
    ].nunique()
)


print(
    "Imágenes repetidas:",
    sample_df[
        "global_embedding_index"
    ].duplicated().sum()
)


print(
    "\nDistribución por zona:"
)

print(
    sample_df[
        "selection_zone"
    ]
    .value_counts()
)


# ============================================================
# DISTRIBUCIÓN POR REGIÓN
# ============================================================

region_check = (
    sample_df
    .groupby("region")
    .size()
    .reset_index(
        name="selected"
    )
)


print(
    "\nDistribución por región:"
)

display(
    region_check
)


# ============================================================
# DISTRIBUCIÓN POR CLUSTER
# ============================================================

cluster_check = (
    sample_df
    .groupby(
        ["region", "cluster"]
    )
    .size()
    .reset_index(
        name="selected"
    )
)


cluster_check = cluster_check.merge(

    allocation_df[
        [
            "region",
            "cluster",
            "allocation"
        ]
    ],

    on=[
        "region",
        "cluster"
    ],

    how="left"
)


cluster_check[
    "difference"
] = (
    cluster_check["selected"] -
    cluster_check["allocation"]
)


print(
    "\nValidación por cluster:"
)

display(
    cluster_check
)


# ============================================================
# ESTUDIOS CON MÁS DE UNA IMAGEN
# ============================================================

study_counts = (
    sample_df[
        "study_id"
    ]
    .value_counts()
)


print(
    "\nEstudios con >1 imagen:",
    (study_counts > 1).sum()
)


print(
    "Máximo de imágenes "
    "seleccionadas de un estudio:",
    study_counts.max()
)


# ============================================================
# GUARDAR
# ============================================================

FULL_CLUSTER_FILE = (
    OUTPUT_DIR /
    "mura_abnormal_train_clustered.csv"
)

SAMPLE_FILE = (
    OUTPUT_DIR /
    "mura_annotation_sample_2000.csv"
)


clustered_df.to_csv(
    FULL_CLUSTER_FILE,
    index=False
)

sample_df.to_csv(
    SAMPLE_FILE,
    index=False
)


print(
    "\nArchivos guardados:"
)

print(
    FULL_CLUSTER_FILE
)

print(
    SAMPLE_FILE
)

print(
    "\n✅ Selección terminada."
)

Procesando XR_ELBOW | K=6
Procesando XR_FINGER | K=6
Procesando XR_FOREARM | K=6
Procesando XR_HAND | K=6
Procesando XR_HUMERUS | K=4
Procesando XR_SHOULDER | K=6
Procesando XR_WRIST | K=9

VALIDACIÓN DE LA MUESTRA

Total seleccionado: 2000
Imágenes únicas: 2000
Estudios únicos: 1980
Imágenes repetidas: 0

Distribución por zona:
selection_zone
central         1200
intermediate     410
peripheral       390
Name: count, dtype: int64

Distribución por región:


,region,selected
0,XR_ELBOW,278
1,XR_FINGER,275
2,XR_FOREARM,187
3,XR_HAND,243
4,XR_HUMERUS,183
5,XR_SHOULDER,423
6,XR_WRIST,411



Validación por cluster:


,region,cluster,selected,allocation,difference
0,XR_ELBOW,0,58,58,0
1,XR_ELBOW,1,67,67,0
2,XR_ELBOW,2,41,41,0
3,XR_ELBOW,3,14,14,0
4,XR_ELBOW,4,51,51,0
5,XR_ELBOW,5,47,47,0
6,XR_FINGER,0,45,45,0
7,XR_FINGER,1,40,40,0
8,XR_FINGER,2,49,49,0
9,XR_FINGER,3,41,41,0



Estudios con >1 imagen: 9
Máximo de imágenes seleccionadas de un estudio: 6

Archivos guardados:
/content/drive/MyDrive/MURA_TT/annotation_sample/mura_abnormal_train_clustered.csv
/content/drive/MyDrive/MURA_TT/annotation_sample/mura_annotation_sample_2000.csv

✅ Selección terminada.


In [ ]:
from pathlib import Path
import pandas as pd

SAMPLE_FILE = Path(
    "/content/drive/MyDrive/MURA_TT/annotation_sample/"
    "mura_annotation_sample_2000.csv"
)

sample_df = pd.read_csv(SAMPLE_FILE)


# ============================================================
# CONTAR IMÁGENES POR ESTUDIO
# ============================================================

study_counts = (
    sample_df
    .groupby("study_id")
    .size()
    .reset_index(name="n_selected")
)


repeated_studies = (
    study_counts[
        study_counts["n_selected"] > 1
    ]
    .sort_values(
        "n_selected",
        ascending=False
    )
)


print("=" * 90)
print("ESTUDIOS REPETIDOS")
print("=" * 90)

display(repeated_studies)


# ============================================================
# DETALLE DE SUS IMÁGENES
# ============================================================

repeated_detail = (
    sample_df[
        sample_df["study_id"].isin(
            repeated_studies["study_id"]
        )
    ]
    [
        [
            "annotation_case_id",
            "region",
            "patient",
            "study",
            "study_id",
            "image_name",
            "cluster",
            "sampling_zone",
            "selection_zone",
            "distance_percentile"
        ]
    ]
    .sort_values(
        [
            "study_id",
            "cluster",
            "distance_percentile"
        ]
    )
)


print("\n" + "=" * 90)
print("DETALLE")
print("=" * 90)

display(
    repeated_detail.reset_index(drop=True)
)


# ============================================================
# RESUMEN
# ============================================================

print("\n" + "=" * 90)
print("RESUMEN")
print("=" * 90)

print(
    "Estudios únicos:",
    sample_df["study_id"].nunique()
)

print(
    "Estudios repetidos:",
    len(repeated_studies)
)

print(
    "Imágenes pertenecientes a estudios repetidos:",
    len(repeated_detail)
)

print(
    "Máximo de imágenes de un mismo estudio:",
    repeated_studies["n_selected"].max()
)


print(
    "\nDistribución del número de imágenes "
    "en estudios repetidos:"
)

print(
    repeated_studies[
        "n_selected"
    ]
    .value_counts()
    .sort_index()
)

ESTUDIOS REPETIDOS


,study_id,n_selected
641,train/XR_FOREARM/patient09153/study1_positive,6
1006,train/XR_HUMERUS/patient02718/study1_positive,4
572,train/XR_FOREARM/patient04136/study1_positive,3
1114,train/XR_HUMERUS/patient02871/study1_positive,3
1012,train/XR_HUMERUS/patient02724/study1_positive,3
1728,train/XR_WRIST/patient06606/study1_positive,3
1969,train/XR_WRIST/patient07402/study1_positive,3
996,train/XR_HUMERUS/patient02704/study1_positive,2
632,train/XR_FOREARM/patient09142/study1_positive,2



DETALLE


,annotation_case_id,region,patient,study,study_id,image_name,cluster,sampling_zone,selection_zone,distance_percentile
0,MURA_TT_1216,XR_FOREARM,patient04136,study1_positive,train/XR_FOREARM/patient04136/study1_positive,image5.png,4,intermediate,intermediate,0.571429
1,MURA_TT_0534,XR_FOREARM,patient04136,study1_positive,train/XR_FOREARM/patient04136/study1_positive,image3.png,4,intermediate,intermediate,0.714286
2,MURA_TT_1765,XR_FOREARM,patient04136,study1_positive,train/XR_FOREARM/patient04136/study1_positive,image1.png,4,peripheral,peripheral,0.928571
3,MURA_TT_0736,XR_FOREARM,patient09142,study1_positive,train/XR_FOREARM/patient09142/study1_positive,image2.png,4,central,central,0.500000
4,MURA_TT_0592,XR_FOREARM,patient09142,study1_positive,train/XR_FOREARM/patient09142/study1_positive,image1.png,4,peripheral,peripheral,0.857143
5,MURA_TT_1904,XR_FOREARM,patient09153,study1_positive,train/XR_FOREARM/patient09153/study1_positive,image6.png,4,central,central,0.071429
6,MURA_TT_0449,XR_FOREARM,patient09153,study1_positive,train/XR_FOREARM/patient09153/study1_positive,image7.png,4,central,central,0.142857
7,MURA_TT_0683,XR_FOREARM,patient09153,study1_positive,train/XR_FOREARM/patient09153/study1_positive,image3.png,4,central,central,0.214286
8,MURA_TT_1459,XR_FOREARM,patient09153,study1_positive,train/XR_FOREARM/patient09153/study1_positive,image2.png,4,central,central,0.285714
9,MURA_TT_0699,XR_FOREARM,patient09153,study1_positive,train/XR_FOREARM/patient09153/study1_positive,image5.png,4,central,central,0.357143



RESUMEN
Estudios únicos: 1980
Estudios repetidos: 9
Imágenes pertenecientes a estudios repetidos: 29
Máximo de imágenes de un mismo estudio: 6

Distribución del número de imágenes en estudios repetidos:
n_selected
2    2
3    5
4    1
6    1
Name: count, dtype: int64


In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd

# ============================================================
# 1. MONTAR GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")


# ============================================================
# 2. RUTAS
# ============================================================

MURA_ROOT = Path(
    "/content/drive/MyDrive/MURA-v1.1"
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MURA_TT"
)

SAMPLE_FILE = (
    PROJECT_ROOT /
    "annotation_sample" /
    "mura_annotation_sample_2000.csv"
)

CLUSTERED_FILE = (
    PROJECT_ROOT /
    "annotation_sample" /
    "mura_abnormal_train_clustered.csv"
)


# ============================================================
# 3. RECUPERAR RESULTADOS
# ============================================================

sample_df = pd.read_csv(
    SAMPLE_FILE
)

clustered_df = pd.read_csv(
    CLUSTERED_FILE
)


# ============================================================
# 4. COMPROBAR
# ============================================================

print("=" * 70)
print("RECUPERACIÓN DEL PROYECTO")
print("=" * 70)

print(
    "\nMURA existe:",
    MURA_ROOT.exists()
)

print(
    "CSV muestra existe:",
    SAMPLE_FILE.exists()
)

print(
    "CSV clustering existe:",
    CLUSTERED_FILE.exists()
)

print(
    "\nImágenes seleccionadas:",
    len(sample_df)
)

print(
    "Estudios:",
    sample_df["study_id"].nunique()
)

print(
    "Regiones:",
    sample_df["region"].nunique()
)

print(
    "Clusters:",
    sample_df.groupby(
        ["region", "cluster"]
    ).ngroups
)

print("\n✅ Proyecto recuperado.")

Mounted at /content/drive
RECUPERACIÓN DEL PROYECTO

MURA existe: True
CSV muestra existe: True
CSV clustering existe: True

Imágenes seleccionadas: 2000
Estudios: 1980
Regiones: 7
Clusters: 43

✅ Proyecto recuperado.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ============================================================
# CONFIGURACIÓN
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MURA_TT"
)

AUDIT_DIR = (
    PROJECT_ROOT /
    "annotation_sample" /
    "visual_audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RANDOM_STATE = 42

ZONES = [
    "central",
    "intermediate",
    "peripheral"
]


# ============================================================
# FUNCIÓN PARA LOCALIZAR LA IMAGEN ORIGINAL
# ============================================================

def resolve_image_path(row):

    # Primero intentamos usar image_path si existe
    if "image_path" in row.index:

        p = Path(str(row["image_path"]))

        if p.exists():
            return p

    # Después reconstruimos usando image_path_csv
    if "image_path_csv" in row.index:

        relative = str(
            row["image_path_csv"]
        )

        # Los CSV originales pueden contener:
        # MURA-v1.1/train/...
        relative = relative.replace(
            "MURA-v1.1/",
            ""
        )

        p = (
            Path(
                "/content/drive/MyDrive/MURA-v1.1"
            ) /
            relative
        )

        if p.exists():
            return p

    # Último intento:
    # region / patient / study / image
    p = (
        Path(
            "/content/drive/MyDrive/MURA-v1.1/train"
        ) /
        str(row["region"]) /
        str(row["patient"]) /
        str(row["study"]) /
        str(row["image_name"])
    )

    return p


# ============================================================
# ELEGIR CASOS PARA AUDITORÍA
# ============================================================

audit_rows = []

for (region, cluster), group in (
    sample_df
    .groupby(
        ["region", "cluster"]
    )
):

    for zone in ZONES:

        candidates = group[
            group["selection_zone"] == zone
        ]

        if len(candidates) == 0:
            continue

        selected = candidates.sample(
            n=1,
            random_state=(
                RANDOM_STATE +
                int(cluster)
            )
        )

        audit_rows.append(
            selected.iloc[0]
        )


audit_df = pd.DataFrame(
    audit_rows
).reset_index(drop=True)


# ============================================================
# RESOLVER RUTAS
# ============================================================

audit_df[
    "audit_image_path"
] = audit_df.apply(
    resolve_image_path,
    axis=1
)


audit_df[
    "audit_exists"
] = audit_df[
    "audit_image_path"
].apply(
    lambda x: Path(x).exists()
)


print("=" * 80)
print("AUDITORÍA VISUAL")
print("=" * 80)

print(
    "\nCasos seleccionados:",
    len(audit_df)
)

print(
    "Archivos encontrados:",
    audit_df[
        "audit_exists"
    ].sum()
)

print(
    "Archivos faltantes:",
    (~audit_df[
        "audit_exists"
    ]).sum()
)


# ============================================================
# GENERAR UN MOSAICO POR REGIÓN
# ============================================================

for region in sorted(
    audit_df["region"].unique()
):

    region_df = (
        audit_df[
            audit_df["region"] == region
        ]
        .sort_values(
            [
                "cluster",
                "selection_zone"
            ]
        )
        .copy()
    )

    clusters = sorted(
        region_df[
            "cluster"
        ].unique()
    )

    nrows = len(clusters)
    ncols = 3

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(
            10,
            max(3, nrows * 3.2)
        )
    )

    if nrows == 1:
        axes = np.array(
            [axes]
        )

    for ax in axes.flat:
        ax.axis("off")


    for row_i, cluster in enumerate(
        clusters
    ):

        cluster_df = region_df[
            region_df[
                "cluster"
            ] == cluster
        ]

        for col_i, zone in enumerate(
            ZONES
        ):

            ax = axes[
                row_i,
                col_i
            ]

            case = cluster_df[
                cluster_df[
                    "selection_zone"
                ] == zone
            ]

            if len(case) == 0:

                ax.set_title(
                    f"C{cluster} | {zone}\nSIN CASO",
                    fontsize=9
                )

                continue


            case = case.iloc[0]

            path = Path(
                case[
                    "audit_image_path"
                ]
            )


            if not path.exists():

                ax.set_title(
                    f"C{cluster} | {zone}\nNO ENCONTRADA",
                    fontsize=9
                )

                continue


            try:

                img = Image.open(
                    path
                ).convert("L")

                ax.imshow(
                    img,
                    cmap="gray"
                )

                ax.set_title(
                    (
                        f"C{cluster} | {zone}\n"
                        f"{case['annotation_case_id']}"
                    ),
                    fontsize=9
                )

            except Exception as e:

                ax.set_title(
                    (
                        f"C{cluster} | {zone}\n"
                        "ERROR"
                    ),
                    fontsize=9
                )


    fig.suptitle(
        f"{region} — auditoría visual",
        fontsize=14
    )

    plt.tight_layout()

    output_file = (
        AUDIT_DIR /
        f"{region.lower()}_audit.png"
    )

    plt.savefig(
        output_file,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

    plt.close(fig)

    print(
        "Guardado:",
        output_file
    )


# ============================================================
# GUARDAR MANIFIESTO DE AUDITORÍA
# ============================================================

audit_csv = (
    AUDIT_DIR /
    "visual_audit_manifest.csv"
)

audit_df.to_csv(
    audit_csv,
    index=False
)


print("\n" + "=" * 80)

print(
    "Manifiesto:",
    audit_csv
)

print(
    "\n✅ Auditoría visual generada."
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# CONFIGURACIÓN
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MURA_TT"
)

ANNOTATION_DIR = (
    PROJECT_ROOT /
    "specialist_annotation"
)

ANNOTATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


SAMPLE_FILE = (
    PROJECT_ROOT /
    "annotation_sample" /
    "mura_annotation_sample_2000.csv"
)


SPECIALISTS = [
    "SPECIALIST_1",
    "SPECIALIST_2",
    "SPECIALIST_3"
]


N_SHARED = 300

RANDOM_STATE = 42


# ============================================================
# CARGAR MUESTRA
# ============================================================

sample_df = pd.read_csv(
    SAMPLE_FILE
)


assert len(sample_df) == 2000

assert (
    sample_df[
        "annotation_case_id"
    ].nunique()
    == 2000
)


# ============================================================
# CREAR ESTRATO
#
# región + cluster + zona
# ============================================================

sample_df[
    "annotation_stratum"
] = (
    sample_df["region"].astype(str)
    + "_C"
    + sample_df["cluster"].astype(str)
    + "_"
    + sample_df["selection_zone"].astype(str)
)


# ============================================================
# ASIGNACIÓN PROPORCIONAL DE LOS 300 COMPARTIDOS
# ENTRE ESTRATOS
# ============================================================

stratum_counts = (
    sample_df[
        "annotation_stratum"
    ]
    .value_counts()
    .rename_axis(
        "annotation_stratum"
    )
    .reset_index(
        name="n_cases"
    )
)


raw = (
    stratum_counts["n_cases"]
    / len(sample_df)
    * N_SHARED
)


stratum_counts[
    "shared_quota"
] = np.floor(
    raw
).astype(int)


# ============================================================
# GARANTIZAR AL MENOS 1 CASO POR ESTRATO
# CUANDO SEA POSIBLE
# ============================================================

stratum_counts.loc[
    (
        (stratum_counts["n_cases"] > 0)
        &
        (stratum_counts["shared_quota"] == 0)
    ),
    "shared_quota"
] = 1


# ============================================================
# AJUSTAR HASTA EXACTAMENTE 300
# ============================================================

current = int(
    stratum_counts[
        "shared_quota"
    ].sum()
)


# Si nos pasamos, quitamos de los estratos
# proporcionalmente más sobrerrepresentados
while current > N_SHARED:

    removable = stratum_counts[
        stratum_counts[
            "shared_quota"
        ] > 1
    ].copy()

    removable[
        "ratio"
    ] = (
        removable[
            "shared_quota"
        ]
        /
        removable[
            "n_cases"
        ]
    )

    idx = (
        removable[
            "ratio"
        ]
        .idxmax()
    )

    stratum_counts.loc[
        idx,
        "shared_quota"
    ] -= 1

    current -= 1


# Si faltan casos, añadimos a los estratos
# más infrarrepresentados
while current < N_SHARED:

    expandable = stratum_counts[
        stratum_counts[
            "shared_quota"
        ]
        <
        stratum_counts[
            "n_cases"
        ]
    ].copy()

    expandable[
        "ratio"
    ] = (
        expandable[
            "shared_quota"
        ]
        /
        expandable[
            "n_cases"
        ]
    )

    idx = (
        expandable[
            "ratio"
        ]
        .idxmin()
    )

    stratum_counts.loc[
        idx,
        "shared_quota"
    ] += 1

    current += 1


# ============================================================
# SELECCIONAR LOS CASOS COMPARTIDOS
# ============================================================

shared_parts = []


for _, row in stratum_counts.iterrows():

    stratum = row[
        "annotation_stratum"
    ]

    quota = int(
        row[
            "shared_quota"
        ]
    )


    candidates = sample_df[
        sample_df[
            "annotation_stratum"
        ] == stratum
    ]


    chosen = candidates.sample(
        n=quota,
        random_state=(
            RANDOM_STATE
            +
            sum(
                ord(c)
                for c in stratum
            )
        )
    )


    shared_parts.append(
        chosen
    )


shared_df = pd.concat(
    shared_parts,
    ignore_index=True
)


assert len(shared_df) == N_SHARED


shared_ids = set(
    shared_df[
        "annotation_case_id"
    ]
)


# ============================================================
# CASOS EXCLUSIVOS
# ============================================================

exclusive_df = sample_df[
    ~sample_df[
        "annotation_case_id"
    ].isin(shared_ids)
].copy()


assert len(exclusive_df) == 1700


# ============================================================
# REPARTIR LOS 1700 ENTRE LOS 3 ESPECIALISTAS
#
# Hacemos round-robin dentro de cada región/cluster/zona
# para que cada especialista reciba una distribución similar.
# ============================================================

rng = np.random.default_rng(
    RANDOM_STATE
)


exclusive_assignments = []


# Mantener contador global para equilibrar carga
specialist_counts = {
    s: 0
    for s in SPECIALISTS
}


for stratum, group in (
    exclusive_df
    .groupby(
        "annotation_stratum"
    )
):

    group = group.copy()

    indices = group.index.to_numpy()

    rng.shuffle(indices)


    for idx in indices:

        # Elegimos al especialista que actualmente
        # tenga menor carga total.
        min_count = min(
            specialist_counts.values()
        )

        available = [
            s
            for s in SPECIALISTS
            if specialist_counts[s]
            == min_count
        ]


        # Si hay empate, elección reproducible
        specialist = available[
            rng.integers(
                0,
                len(available)
            )
        ]


        exclusive_assignments.append(
            {
                "annotation_case_id":
                    sample_df.loc[
                        idx,
                        "annotation_case_id"
                    ],

                "assigned_specialist":
                    specialist
            }
        )


        specialist_counts[
            specialist
        ] += 1


exclusive_assignment_df = pd.DataFrame(
    exclusive_assignments
)


# ============================================================
# TABLA MAESTRA DE ASIGNACIONES
# ============================================================

assignment_rows = []


# Casos compartidos:
# crear una fila por especialista
for _, row in shared_df.iterrows():

    for specialist in SPECIALISTS:

        assignment_rows.append(
            {
                "annotation_case_id":
                    row[
                        "annotation_case_id"
                    ],

                "assigned_specialist":
                    specialist,

                "assignment_type":
                    "shared"
            }
        )


# Casos exclusivos:
for _, row in (
    exclusive_assignment_df.iterrows()
):

    assignment_rows.append(
        {
            "annotation_case_id":
                row[
                    "annotation_case_id"
                ],

            "assigned_specialist":
                row[
                    "assigned_specialist"
                ],

            "assignment_type":
                "exclusive"
        }
    )


assignment_df = pd.DataFrame(
    assignment_rows
)


# ============================================================
# AÑADIR METADATOS SOLO PARA AUDITORÍA INTERNA
# ============================================================

internal_assignment_df = (
    assignment_df
    .merge(
        sample_df,
        on="annotation_case_id",
        how="left"
    )
)


# ============================================================
# VALIDACIONES
# ============================================================

print("=" * 85)
print("ASIGNACIÓN A ESPECIALISTAS")
print("=" * 85)


print(
    "\nCasos originales:",
    sample_df[
        "annotation_case_id"
    ].nunique()
)


print(
    "Casos compartidos:",
    len(shared_df)
)


print(
    "Casos exclusivos:",
    len(exclusive_df)
)


print(
    "Evaluaciones totales:",
    len(assignment_df)
)


print(
    "\nCarga por especialista:"
)

load = (
    assignment_df
    .groupby(
        [
            "assigned_specialist",
            "assignment_type"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


load[
    "TOTAL"
] = load.sum(
    axis=1
)


display(load)


# ============================================================
# COMPROBAR NÚMERO DE EVALUADORES POR CASO
# ============================================================

raters_per_case = (
    assignment_df
    .groupby(
        "annotation_case_id"
    )[
        "assigned_specialist"
    ]
    .nunique()
)


print(
    "\nCasos con 1 evaluador:",
    (raters_per_case == 1).sum()
)


print(
    "Casos con 3 evaluadores:",
    (raters_per_case == 3).sum()
)


print(
    "Otros:",
    (
        ~raters_per_case.isin(
            [1, 3]
        )
    ).sum()
)


# ============================================================
# DISTRIBUCIÓN DE LOS 300 COMPARTIDOS POR REGIÓN
# ============================================================

shared_region = (
    shared_df
    .groupby("region")
    .size()
    .reset_index(
        name="shared_cases"
    )
)


print(
    "\nCasos compartidos por región:"
)

display(
    shared_region
)


# ============================================================
# DISTRIBUCIÓN TOTAL POR ESPECIALISTA Y REGIÓN
# ============================================================

specialist_region = (
    internal_assignment_df
    .groupby(
        [
            "assigned_specialist",
            "region"
        ]
    )
    .size()
    .reset_index(
        name="n_cases"
    )
)


print(
    "\nDistribución por especialista/región:"
)

display(
    specialist_region
    .pivot(
        index="region",
        columns="assigned_specialist",
        values="n_cases"
    )
    .fillna(0)
    .astype(int)
)


# ============================================================
# GUARDAR
# ============================================================

assignment_df.to_csv(
    ANNOTATION_DIR /
    "specialist_assignments.csv",
    index=False
)


internal_assignment_df.to_csv(
    ANNOTATION_DIR /
    "specialist_assignments_internal.csv",
    index=False
)


shared_df.to_csv(
    ANNOTATION_DIR /
    "shared_300_internal.csv",
    index=False
)


print(
    "\nArchivos guardados en:"
)

print(
    ANNOTATION_DIR
)

print(
    "\n✅ Asignación terminada."
)

ASIGNACIÓN A ESPECIALISTAS

Casos originales: 2000
Casos compartidos: 300
Casos exclusivos: 1700
Evaluaciones totales: 2600

Carga por especialista:


assignment_type,exclusive,shared,TOTAL
assigned_specialist,,,
SPECIALIST_1,567,300,867
SPECIALIST_2,566,300,866
SPECIALIST_3,567,300,867



Casos con 1 evaluador: 1700
Casos con 3 evaluadores: 300
Otros: 0

Casos compartidos por región:


,region,shared_cases
0,XR_ELBOW,43
1,XR_FINGER,40
2,XR_FOREARM,30
3,XR_HAND,37
4,XR_HUMERUS,27
5,XR_SHOULDER,59
6,XR_WRIST,64



Distribución por especialista/región:


assigned_specialist,SPECIALIST_1,SPECIALIST_2,SPECIALIST_3
region,,,
XR_ELBOW,121,121,122
XR_FINGER,119,118,118
XR_FOREARM,82,83,82
XR_HAND,106,106,105
XR_HUMERUS,79,79,79
XR_SHOULDER,180,180,181
XR_WRIST,180,179,180



Archivos guardados en:
/content/drive/MyDrive/MURA_TT/specialist_annotation

✅ Asignación terminada.


In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import shutil
import hashlib


# ============================================================
# CONFIGURACIÓN
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MURA_TT"
)

SOURCE_SAMPLE = (
    PROJECT_ROOT /
    "annotation_sample" /
    "mura_annotation_sample_2000.csv"
)

FINAL_DIR = (
    PROJECT_ROOT /
    "sampling_final"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RANDOM_STATE = 42

N_TARGET = 2000


K_FINAL = {
    "XR_ELBOW": 6,
    "XR_FINGER": 6,
    "XR_FOREARM": 6,
    "XR_HAND": 6,
    "XR_HUMERUS": 4,
    "XR_SHOULDER": 6,
    "XR_WRIST": 9
}


# ============================================================
# 1. CARGAR MUESTRA DEFINITIVA
# ============================================================

sample_df = pd.read_csv(
    SOURCE_SAMPLE
)


print("=" * 80)
print("CIERRE DEL MUESTREO")
print("=" * 80)

print(
    "\nArchivo fuente:",
    SOURCE_SAMPLE
)

print(
    "Casos cargados:",
    len(sample_df)
)


# ============================================================
# 2. VALIDACIONES CRÍTICAS
# ============================================================

assert len(sample_df) == N_TARGET, (
    f"ERROR: se esperaban {N_TARGET} casos "
    f"y hay {len(sample_df)}"
)

assert (
    sample_df[
        "annotation_case_id"
    ].nunique()
    == N_TARGET
), "ERROR: hay annotation_case_id duplicados"


if "global_embedding_index" in sample_df.columns:

    assert (
        sample_df[
            "global_embedding_index"
        ].nunique()
        == N_TARGET
    ), "ERROR: hay imágenes duplicadas"


assert (
    sample_df["region"]
    .nunique()
    == 7
), "ERROR: no aparecen las 7 regiones"


n_clusters = (
    sample_df
    .groupby(
        ["region", "cluster"]
    )
    .ngroups
)

assert n_clusters == 43, (
    f"ERROR: se esperaban 43 clusters "
    f"y aparecen {n_clusters}"
)


print("\nValidaciones críticas: OK")


# ============================================================
# 3. COPIA INTERNA CONGELADA
#
# Conserva TODA la información metodológica.
# Este archivo NO es para los especialistas.
# ============================================================

internal_file = (
    FINAL_DIR /
    "sampling_2000_internal.csv"
)

sample_df.to_csv(
    internal_file,
    index=False
)


# ============================================================
# 4. CREAR RUTA DE LA IMAGEN ORIGINAL
# PARA LA TRANSFERENCIA A JUAN
# ============================================================

def build_original_path(row):

    return str(
        Path(
            "/content/drive/MyDrive/MURA-v1.1/train"
        )
        /
        str(row["region"])
        /
        str(row["patient"])
        /
        str(row["study"])
        /
        str(row["image_name"])
    )


sample_df[
    "source_image_path"
] = sample_df.apply(
    build_original_path,
    axis=1
)


sample_df[
    "source_exists"
] = sample_df[
    "source_image_path"
].apply(
    lambda x: Path(x).exists()
)


missing = int(
    (~sample_df["source_exists"]).sum()
)


print(
    "\nImágenes originales encontradas:",
    sample_df["source_exists"].sum()
)

print(
    "Imágenes originales faltantes:",
    missing
)


assert missing == 0, (
    "ERROR: faltan imágenes originales. "
    "No continuar con la entrega."
)


# ============================================================
# 5. CSV LIMPIO DE TRANSFERENCIA
#
# No contiene:
# - cluster
# - distancia al centroide
# - zona de muestreo
# - etiqueta positive
# - embedding
#
# Juan recibe únicamente la información necesaria
# para localizar cada caso.
# ============================================================

handoff_columns = [
    "annotation_case_id",
    "region",
    "source_image_path"
]


handoff_df = (
    sample_df[
        handoff_columns
    ]
    .copy()
    .sort_values(
        "annotation_case_id"
    )
    .reset_index(drop=True)
)


handoff_file = (
    FINAL_DIR /
    "sampling_2000_handoff.csv"
)


handoff_df.to_csv(
    handoff_file,
    index=False
)


# ============================================================
# 6. RESUMEN POR REGIÓN
# ============================================================

region_summary = (
    sample_df
    .groupby("region")
    .agg(
        selected_images=(
            "annotation_case_id",
            "count"
        ),
        unique_studies=(
            "study_id",
            "nunique"
        ),
        n_clusters=(
            "cluster",
            "nunique"
        )
    )
    .reset_index()
)


region_summary[
    "sample_pct"
] = (
    region_summary[
        "selected_images"
    ]
    /
    len(sample_df)
    * 100
)


region_summary[
    "k_final"
] = region_summary[
    "region"
].map(
    K_FINAL
)


summary_file = (
    FINAL_DIR /
    "sampling_summary.csv"
)


region_summary.to_csv(
    summary_file,
    index=False
)


# ============================================================
# 7. RESUMEN POR CLUSTER
# ============================================================

cluster_summary = (
    sample_df
    .groupby(
        ["region", "cluster"]
    )
    .agg(
        selected_images=(
            "annotation_case_id",
            "count"
        ),
        unique_studies=(
            "study_id",
            "nunique"
        ),
        central=(
            "selection_zone",
            lambda x: (
                x == "central"
            ).sum()
        ),
        intermediate=(
            "selection_zone",
            lambda x: (
                x == "intermediate"
            ).sum()
        ),
        peripheral=(
            "selection_zone",
            lambda x: (
                x == "peripheral"
            ).sum()
        )
    )
    .reset_index()
)


cluster_summary_file = (
    FINAL_DIR /
    "cluster_summary.csv"
)


cluster_summary.to_csv(
    cluster_summary_file,
    index=False
)


# ============================================================
# 8. RESUMEN DE ZONAS
# ============================================================

zone_summary = (
    sample_df[
        "selection_zone"
    ]
    .value_counts()
    .rename_axis(
        "selection_zone"
    )
    .reset_index(
        name="n_images"
    )
)


zone_summary[
    "percentage"
] = (
    zone_summary[
        "n_images"
    ]
    /
    len(sample_df)
    * 100
)


zone_file = (
    FINAL_DIR /
    "sampling_zone_summary.csv"
)


zone_summary.to_csv(
    zone_file,
    index=False
)


# ============================================================
# 9. HASH SHA-256 DEL MANIFIESTO INTERNO
#
# Esto permite comprobar después que el archivo
# congelado no fue modificado.
# ============================================================

def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            sha.update(block)

    return sha.hexdigest()


internal_hash = sha256_file(
    internal_file
)


# ============================================================
# 10. README DE TRAZABILIDAD
# ============================================================

zone_counts = (
    sample_df[
        "selection_zone"
    ]
    .value_counts()
)


readme_text = f"""
MURA_TT - RESUMEN DEL MUESTREO FINAL
====================================

Fecha de congelamiento:
{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

OBJETIVO
--------
Construir un subconjunto enriquecido de radiografías anormales
del conjunto de entrenamiento de MURA para posterior revisión
y anotación por especialistas.

UNIVERSO DE MUESTREO
--------------------
Dataset: MURA-v1.1
Split utilizado para clustering/muestreo: train
Clase utilizada: abnormal/positive

Radiografías anormales de train: 14,873
Estudios anormales de train: 5,177

El conjunto valid oficial de MURA NO fue utilizado para
clustering ni para seleccionar este subconjunto.

REPRESENTACIÓN VISUAL
---------------------
Extractor de características:
DenseNet121 preentrenada en ImageNet.

La capa clasificadora fue retirada y se utilizaron embeddings
visuales de 1024 dimensiones.

La red fue utilizada únicamente como extractor de
características visuales, no como clasificador clínico.

REDUCCIÓN DE DIMENSIONALIDAD
----------------------------
Se aplicó StandardScaler y PCA de forma independiente para
cada región anatómica.

Se conservaron componentes suficientes para explicar
aproximadamente el 95% de la varianza.

CLUSTERING
----------
Algoritmo: K-Means
Semilla principal: {RANDOM_STATE}

K final por región:

XR_ELBOW:    6
XR_FINGER:   6
XR_FOREARM:  6
XR_HAND:     6
XR_HUMERUS:  4
XR_SHOULDER: 6
XR_WRIST:    9

Total de clusters: 43

La elección de K consideró:
- Silhouette Score
- Davies-Bouldin Index
- Calinski-Harabasz Index
- distribución de tamaños de cluster
- estabilidad mediante Adjusted Rand Index (ARI)
- inspección visual de representantes

Los clusters se utilizaron para representar diversidad visual.
No deben interpretarse como categorías diagnósticas.

ESTRATEGIA DE ASIGNACIÓN
------------------------
Tamaño objetivo del subconjunto: 2,000 radiografías.

Se utilizó una asignación híbrida por región que combinó:
- representación proporcional al tamaño de la región
- cobertura equilibrada entre regiones

Dentro de cada región se garantizó cobertura de los clusters
y posteriormente se distribuyó el resto de manera proporcional.

MUESTREO DENTRO DE CLUSTERS
---------------------------
Las imágenes fueron ordenadas según su distancia al centroide.

Zonas consideradas:

Central:
percentil de distancia 0-50.

Intermedia:
percentil de distancia 50-80.

Periférica:
percentil de distancia 80-95.

Extrema:
percentil 95-100, no priorizada para el muestreo ordinario.

Distribución final:

Central:
{zone_counts.get("central", 0)} imágenes
({zone_counts.get("central", 0) / len(sample_df) * 100:.2f}%)

Intermedia:
{zone_counts.get("intermediate", 0)} imágenes
({zone_counts.get("intermediate", 0) / len(sample_df) * 100:.2f}%)

Periférica:
{zone_counts.get("peripheral", 0)} imágenes
({zone_counts.get("peripheral", 0) / len(sample_df) * 100:.2f}%)

RESULTADO FINAL
---------------
Radiografías seleccionadas: {len(sample_df)}

Estudios únicos:
{sample_df["study_id"].nunique()}

Imágenes duplicadas:
{sample_df["annotation_case_id"].duplicated().sum()}

Regiones anatómicas:
{sample_df["region"].nunique()}

Clusters representados:
{n_clusters}

Se priorizó una imagen por estudio siempre que fue posible.

Los pocos estudios con múltiples imágenes seleccionadas
corresponden a clusters visuales pequeños en los que no había
suficientes estudios independientes para cubrir la cuota
establecida.

AUDITORÍA VISUAL
----------------
Se realizó una auditoría visual estratificada con ejemplos
centrales, intermedios y periféricos de los clusters finales.

La auditoría se utilizó para comprobar la coherencia visual
del muestreo y verificar que los casos periféricos no
correspondieran sistemáticamente a imágenes inutilizables.

INTERPRETACIÓN
--------------
Este subconjunto NO pretende preservar la prevalencia natural
de los hallazgos de MURA.

Se trata de una muestra enriquecida orientada a maximizar
cobertura anatómica y diversidad visual para la posterior
revisión por especialistas.

ARCHIVOS
--------
sampling_2000_internal.csv
    Manifiesto interno completo del muestreo.
    No entregar a los especialistas.

sampling_2000_handoff.csv
    Archivo simplificado para transferencia al responsable
    de la etapa posterior.

sampling_summary.csv
    Resumen del muestreo por región.

cluster_summary.csv
    Resumen de los 43 clusters seleccionados.

sampling_zone_summary.csv
    Distribución central/intermedia/periférica.

INTEGRIDAD
----------
SHA-256 de sampling_2000_internal.csv:

{internal_hash}
"""


readme_file = (
    FINAL_DIR /
    "README_sampling.txt"
)


with open(
    readme_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        readme_text.strip()
        + "\n"
    )


# ============================================================
# 11. VALIDACIÓN FINAL
# ============================================================

print("\n" + "=" * 80)
print("RESULTADO FINAL")
print("=" * 80)


print(
    "\nRadiografías:",
    len(sample_df)
)

print(
    "Estudios únicos:",
    sample_df[
        "study_id"
    ].nunique()
)

print(
    "Regiones:",
    sample_df[
        "region"
    ].nunique()
)

print(
    "Clusters:",
    n_clusters
)

print(
    "Duplicados:",
    sample_df[
        "annotation_case_id"
    ].duplicated().sum()
)


print(
    "\nDistribución por zona:"
)

display(
    zone_summary
)


print(
    "\nDistribución por región:"
)

display(
    region_summary
)


print(
    "\nSHA-256:"
)

print(
    internal_hash
)


print("\n" + "=" * 80)
print("ARCHIVOS GENERADOS")
print("=" * 80)


for file in [
    internal_file,
    handoff_file,
    summary_file,
    cluster_summary_file,
    zone_file,
    readme_file
]:

    print(
        "✓",
        file
    )


print(
    "\n✅ MUESTREO FINAL CONGELADO."
)

CIERRE DEL MUESTREO

Archivo fuente: /content/drive/MyDrive/MURA_TT/annotation_sample/mura_annotation_sample_2000.csv
Casos cargados: 2000

Validaciones críticas: OK

Imágenes originales encontradas: 2000
Imágenes originales faltantes: 0

RESULTADO FINAL

Radiografías: 2000
Estudios únicos: 1980
Regiones: 7
Clusters: 43
Duplicados: 0

Distribución por zona:


,selection_zone,n_images,percentage
0,central,1200,60.0
1,intermediate,410,20.5
2,peripheral,390,19.5



Distribución por región:


,region,selected_images,unique_studies,n_clusters,sample_pct,k_final
0,XR_ELBOW,278,278,6,13.90,6
1,XR_FINGER,275,275,6,13.75,6
2,XR_FOREARM,187,179,6,9.35,6
3,XR_HAND,243,243,6,12.15,6
4,XR_HUMERUS,183,175,4,9.15,4
5,XR_SHOULDER,423,423,6,21.15,6
6,XR_WRIST,411,407,9,20.55,9



SHA-256:
6e67b8ceb4f6fba5a689a1f7480d186e24371b208fede1417203bb13b6cc1857

ARCHIVOS GENERADOS
✓ /content/drive/MyDrive/MURA_TT/sampling_final/sampling_2000_internal.csv
✓ /content/drive/MyDrive/MURA_TT/sampling_final/sampling_2000_handoff.csv
✓ /content/drive/MyDrive/MURA_TT/sampling_final/sampling_summary.csv
✓ /content/drive/MyDrive/MURA_TT/sampling_final/cluster_summary.csv
✓ /content/drive/MyDrive/MURA_TT/sampling_final/sampling_zone_summary.csv
✓ /content/drive/MyDrive/MURA_TT/sampling_final/README_sampling.txt

✅ MUESTREO FINAL CONGELADO.
